# PUBLIC DATASET ACQUISITION AND VERIFICATION

In [1]:
import os
import pandas as pd
import numpy as np

file_path = r"F:\.0 Work\3931\XLPE_Aging_Dataset.csv"

print("=" * 75)
print("STEP 1: XLPE DATASET ACQUISITION AND VERIFICATION")
print("=" * 75)

if not os.path.exists(file_path):
    raise FileNotFoundError(
        f"Dataset not found at:\n{file_path}\n"
        "Please verify the file path."
    )

print("\n[1] DATASET FILE VERIFICATION")
print("-" * 75)
print("File path:", file_path)
print("File exists: Yes")
print("File size: {:.2f} MB".format(
    os.path.getsize(file_path) / (1024 ** 2)
))

df = pd.read_csv(file_path, low_memory=False)

print("\n[2] DATASET LOADING")
print("-" * 75)
print("Dataset loaded successfully.")
print("Number of records:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\n[3] COLUMN NAMES")
print("-" * 75)

for index, column in enumerate(df.columns, start=1):
    print(f"{index:02d}. {column}")

print("\n[4] FIRST FIVE RECORDS")
print("-" * 75)
print(df.head().to_string())

print("\n[5] DATA TYPES")
print("-" * 75)
print(df.dtypes.to_string())

print("\n[6] DATASET INFORMATION")
print("-" * 75)
print("Dataset shape:", df.shape)
print("Total cells:", df.size)
print("Memory usage: {:.2f} MB".format(
    df.memory_usage(deep=True).sum() / (1024 ** 2)
))

print("\n[7] MISSING VALUE ANALYSIS")
print("-" * 75)

missing_count = df.isnull().sum()
missing_percentage = (missing_count / len(df)) * 100

missing_summary = pd.DataFrame({
    "Missing_Count": missing_count,
    "Missing_Percentage": missing_percentage.round(2)
})

missing_summary = missing_summary.sort_values(
    by="Missing_Count",
    ascending=False
)

print(missing_summary.to_string())

print("\nTotal missing values:", int(df.isnull().sum().sum()))

print("\n[8] DUPLICATE RECORD ANALYSIS")
print("-" * 75)

duplicate_count = int(df.duplicated().sum())

print("Duplicate records:", duplicate_count)
print("Unique records:", len(df) - duplicate_count)

print("\n[9] AGING-TIME COLUMN IDENTIFICATION")
print("-" * 75)

time_keywords = [
    "time", "hour", "hours", "aging", "age",
    "duration", "period", "timestamp"
]

time_columns = [
    col for col in df.columns
    if any(keyword in str(col).lower()
           for keyword in time_keywords)
]

if time_columns:
    print("Potential aging-time columns:", time_columns)

    for col in time_columns:
        print(f"\nColumn: {col}")
        print("Data type:", df[col].dtype)
        print("Unique values:", df[col].nunique())
        print("Sample values:")
        print(df[col].dropna().unique()[:20])
else:
    print("No obvious aging-time column was identified.")
    print("Check the column names manually.")

print("\n[10] NUMERICAL FEATURE SUMMARY")
print("-" * 75)

numeric_df = df.select_dtypes(include=np.number)

if not numeric_df.empty:
    print(numeric_df.describe().T.to_string())
else:
    print("No numerical columns were identified.")

print("\n[11] CATEGORICAL FEATURE SUMMARY")
print("-" * 75)

categorical_df = df.select_dtypes(
    include=["object", "category", "bool"]
)

if not categorical_df.empty:
    for col in categorical_df.columns:
        print(f"\nColumn: {col}")
        print("Unique categories:", categorical_df[col].nunique())
        print("Top values:")
        print(categorical_df[col].value_counts(dropna=False).head(10))
else:
    print("No categorical columns were identified.")

print("\n[12] POTENTIAL TARGET-VARIABLE IDENTIFICATION")
print("-" * 75)

target_keywords = [
    "degradation", "di", "failure",
    "class", "rul", "remaining", "lifetime",
    "life", "target"
]

potential_targets = [
    col for col in df.columns
    if any(keyword in str(col).lower()
           for keyword in target_keywords)
]

if potential_targets:
    print("Potential target columns:", potential_targets)

    for col in potential_targets:
        print(f"\nColumn: {col}")
        print("Data type:", df[col].dtype)
        print("Unique values:", df[col].nunique())
        print("Sample values:")
        print(df[col].dropna().unique()[:20])
else:
    print("No obvious target columns were identified.")
    print("Target variables must be verified manually.")

print("\n[13] INFINITE VALUE ANALYSIS")
print("-" * 75)

if not numeric_df.empty:
    infinite_counts = np.isinf(numeric_df).sum()
    print(infinite_counts[infinite_counts > 0].to_string()
          if (infinite_counts > 0).any()
          else "No infinite values found.")
else:
    print("No numerical columns available for this check.")

output_folder = os.path.dirname(file_path)

missing_summary.to_csv(
    os.path.join(output_folder, "step1_missing_value_report.csv")
)

column_summary = pd.DataFrame({
    "Column_Name": df.columns,
    "Data_Type": [str(df[col].dtype) for col in df.columns],
    "Non_Null_Count": [int(df[col].notna().sum()) for col in df.columns],
    "Unique_Values": [int(df[col].nunique(dropna=True))
                      for col in df.columns]
})

column_summary.to_csv(
    os.path.join(output_folder, "step1_column_summary.csv"),
    index=False
)

print("\n" + "=" * 75)
print("STEP 1 COMPLETED")
print("=" * 75)

print("Dataset records:", len(df))
print("Dataset features/columns:", len(df.columns))
print("Missing values:", int(df.isnull().sum().sum()))
print("Duplicate records:", duplicate_count)
print("Potential aging-time columns:", time_columns)
print("Potential target columns:", potential_targets)

print("\nGenerated reports:")
print(os.path.join(output_folder, "step1_missing_value_report.csv"))
print(os.path.join(output_folder, "step1_column_summary.csv"))

STEP 1: XLPE DATASET ACQUISITION AND VERIFICATION

[1] DATASET FILE VERIFICATION
---------------------------------------------------------------------------
File path: F:\.0 Work\3931\XLPE_Aging_Dataset.csv
File exists: Yes
File size: 0.46 MB

[2] DATASET LOADING
---------------------------------------------------------------------------
Dataset loaded successfully.
Number of records: 1500
Number of columns: 20

[3] COLUMN NAMES
---------------------------------------------------------------------------
01. Sample_ID
02. Aging_Type
03. Aging_Time_hr
04. Diffusion_Coefficient
05. Tensile_Strength_MPa
06. Elongation_at_Break_percent
07. Youngs_Modulus_MPa
08. Contact_Angle_Degree
09. Dielectric_Constant
10. Dielectric_Loss
11. Breakdown_Strength_kVmm
12. Trap_Depth_eV
13. Trap_Density
14. LIBS_Carbon_Intensity
15. LIBS_Oxygen_Intensity
16. LIBS_Sodium_Intensity
17. LIBS_Magnesium_Intensity
18. Degradation_Index
19. Remaining_Useful_Life_hr
20. Failure_Class

[4] FIRST FIVE RECORDS
------

# DATA CLEANING AND PREPROCESSING

In [2]:
import os
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler, LabelEncoder

file_path = r"F:\.0 Work\3931\XLPE_Aging_Dataset.csv"
output_folder = os.path.dirname(file_path)

cleaned_path = os.path.join(
    output_folder, "XLPE_Aging_Cleaned.csv"
)

scaled_path = os.path.join(
    output_folder, "XLPE_Numerical_Features_Scaled.csv"
)

quality_report_path = os.path.join(
    output_folder, "step2_data_quality_report.csv"
)

encoding_report_path = os.path.join(
    output_folder, "step2_categorical_encoding_report.csv"
)

print("=" * 75)
print("STEP 2: DATA CLEANING AND PREPROCESSING")
print("=" * 75)

if not os.path.exists(file_path):
    raise FileNotFoundError(
        f"Dataset not found: {file_path}"
    )

df = pd.read_csv(file_path, low_memory=False)

print("\n[1] ORIGINAL DATASET")
print("-" * 75)
print("Dataset shape:", df.shape)

original_columns = df.columns.tolist()

required_columns = [
    "Sample_ID",
    "Aging_Type",
    "Aging_Time_hr",
    "Degradation_Index",
    "Remaining_Useful_Life_hr",
    "Failure_Class"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {missing_columns}"
    )

print("\n[2] REQUIRED COLUMN VALIDATION")
print("-" * 75)
print("All required columns are present.")

duplicate_count = int(df.duplicated().sum())

if duplicate_count > 0:
    df = df.drop_duplicates().copy()

print("\n[3] DUPLICATE RECORD CHECK")
print("-" * 75)
print("Duplicates detected:", duplicate_count)
print("Records after duplicate removal:", len(df))

df = df.replace(
    ["", " ", "NA", "N/A", "NULL", "null", "None"],
    np.nan
)

numeric_columns = [
    col for col in df.columns
    if col not in ["Aging_Type", "Failure_Class"]
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("\n[4] NUMERICAL COLUMN VALIDATION")
print("-" * 75)
print("Numerical columns checked:", len(numeric_columns))

numeric_df = df[numeric_columns]

infinite_count = int(
    np.isinf(numeric_df.to_numpy(dtype=float)).sum()
)

print("\n[5] INFINITE VALUE CHECK")
print("-" * 75)
print("Infinite numerical values:", infinite_count)

if infinite_count > 0:
    df[numeric_columns] = df[numeric_columns].replace(
        [np.inf, -np.inf], np.nan
    )

missing_before = df.isnull().sum()

print("\n[6] MISSING VALUE CHECK")
print("-" * 75)
print("Total missing cells:", int(missing_before.sum()))

target_columns = [
    "Degradation_Index",
    "Remaining_Useful_Life_hr",
    "Failure_Class"
]

identifier_columns = ["Sample_ID"]

numerical_predictor_columns = [
    col for col in numeric_columns
    if col not in target_columns + identifier_columns
]

for col in numerical_predictor_columns:
    if df[col].isnull().any():
        median_value = df[col].median()

        if pd.isna(median_value):
            raise ValueError(
                f"Column '{col}' contains no usable numerical values."
            )

        df[col] = df[col].fillna(median_value)

critical_columns = [
    "Sample_ID",
    "Aging_Type",
    "Aging_Time_hr",
    "Degradation_Index",
    "Remaining_Useful_Life_hr",
    "Failure_Class"
]

rows_before = len(df)

df = df.dropna(subset=critical_columns).copy()

rows_removed = rows_before - len(df)

print("\n[7] MISSING VALUE HANDLING")
print("-" * 75)
print("Rows removed due to missing critical values:", rows_removed)
print("Rows retained:", len(df))

for col in ["Aging_Type", "Failure_Class"]:
    df[col] = df[col].astype("string").str.strip()

    if df[col].isnull().any() or (df[col] == "").any():
        raise ValueError(
            f"Missing or blank category detected in '{col}'."
        )

print("\n[8] CATEGORICAL VALUE CHECK")
print("-" * 75)

for col in ["Aging_Type", "Failure_Class"]:
    print(f"\n{col} distribution:")
    print(df[col].value_counts().to_string())

if (df["Aging_Time_hr"] < 0).any():
    raise ValueError(
        "Negative aging times were found. Inspect the dataset."
    )

print("\n[9] AGING-TIME VALIDATION")
print("-" * 75)
print("Available aging times:")
print(sorted(df["Aging_Time_hr"].unique().tolist()))

print("\n[10] RECORD AND TEMPORAL STRUCTURE")
print("-" * 75)

print("Unique Sample_ID values:", df["Sample_ID"].nunique())
print("Total records:", len(df))

print("\nRecords per Sample_ID (first 10):")
print(
    df.groupby("Sample_ID")
      .size()
      .head(10)
      .to_string()
)

print("\nRecords per aging type:")
print(df["Aging_Type"].value_counts().to_string())

print("\nRecords per aging time:")
print(
    df["Aging_Time_hr"]
      .value_counts()
      .sort_index()
      .to_string()
)

df = df[original_columns]

df.to_csv(cleaned_path, index=False)

print("\n[11] CLEANED DATASET SAVED")
print("-" * 75)
print("Output:", cleaned_path)
print("Cleaned dataset shape:", df.shape)

encoding_mappings = []

encoded_df = df.copy()

for col in ["Aging_Type", "Failure_Class"]:
    encoder = LabelEncoder()

    encoded_df[col + "_Encoded"] = encoder.fit_transform(
        df[col].astype(str)
    )

    for original_value, encoded_value in zip(
        encoder.classes_,
        encoder.transform(encoder.classes_)
    ):
        encoding_mappings.append({
            "Column": col,
            "Original_Value": original_value,
            "Encoded_Value": int(encoded_value)
        })

encoding_report = pd.DataFrame(encoding_mappings)

encoding_report.to_csv(
    encoding_report_path,
    index=False
)

print("\n[12] CATEGORICAL ENCODING")
print("-" * 75)
print(encoding_report.to_string(index=False))
print("Encoding report:", encoding_report_path)

excluded_columns = [
    "Sample_ID",
    "Degradation_Index",
    "Remaining_Useful_Life_hr",
    "Failure_Class"
]

feature_columns = [
    col for col in df.columns
    if col not in excluded_columns
]

X = df[feature_columns].copy()

X = pd.get_dummies(
    X,
    columns=["Aging_Type"],
    prefix="Aging_Type",
    dtype=int
)


numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

scaler = MinMaxScaler()

X_scaled = X.copy()

X_scaled[numeric_features] = scaler.fit_transform(
    X[numeric_features]
)

X_scaled.to_csv(scaled_path, index=False)

print("\n[13] FEATURE SCALING")
print("-" * 75)
print("Predictor matrix shape:", X.shape)
print("Number of predictor columns:", len(X.columns))
print("Scaled feature file:", scaled_path)

quality_report = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": [str(df[col].dtype) for col in df.columns],
    "Missing_Values": [int(df[col].isnull().sum()) for col in df.columns],
    "Unique_Values": [int(df[col].nunique(dropna=True)) for col in df.columns]
})

quality_report.to_csv(
    quality_report_path,
    index=False
)

print("\n[14] DATA-QUALITY REPORT")
print("-" * 75)
print("Report saved:", quality_report_path)

print("\n" + "=" * 75)
print("STEP 2 COMPLETED")
print("=" * 75)

print("Original records: 1500")
print("Final records:", len(df))
print("Final columns:", len(df.columns))
print("Remaining missing cells:", int(df.isnull().sum().sum()))
print("Remaining duplicate records:", int(df.duplicated().sum()))

print("\nGenerated files:")
print("1.", cleaned_path)
print("2.", scaled_path)
print("3.", quality_report_path)
print("4.", encoding_report_path)

STEP 2: DATA CLEANING AND PREPROCESSING

[1] ORIGINAL DATASET
---------------------------------------------------------------------------
Dataset shape: (1500, 20)

[2] REQUIRED COLUMN VALIDATION
---------------------------------------------------------------------------
All required columns are present.

[3] DUPLICATE RECORD CHECK
---------------------------------------------------------------------------
Duplicates detected: 0
Records after duplicate removal: 1500

[4] NUMERICAL COLUMN VALIDATION
---------------------------------------------------------------------------
Numerical columns checked: 18

[5] INFINITE VALUE CHECK
---------------------------------------------------------------------------
Infinite numerical values: 0

[6] MISSING VALUE CHECK
---------------------------------------------------------------------------
Total missing cells: 0

[7] MISSING VALUE HANDLING
---------------------------------------------------------------------------
Rows removed due to missing cri

# MULTIMODAL FEATURE CONSTRUCTION

In [3]:
import os
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder

input_path = r"F:\.0 Work\3931\XLPE_Aging_Cleaned.csv"

output_folder = os.path.dirname(input_path)

sequence_path = os.path.join(
    output_folder, "step3_XLPE_temporal_sequences.npz"
)

target_path = os.path.join(
    output_folder, "step3_XLPE_targets.csv"
)

sequence_info_path = os.path.join(
    output_folder, "step3_sequence_verification.csv"
)

feature_manifest_path = os.path.join(
    output_folder, "step3_feature_manifest.csv"
)

print("=" * 75)
print("STEP 3: MULTIMODAL FEATURE CONSTRUCTION")
print("AND TEMPORAL SEQUENCE PREPARATION")
print("=" * 75)

if not os.path.exists(input_path):
    raise FileNotFoundError(
        f"Cleaned dataset not found: {input_path}"
    )

df = pd.read_csv(input_path)

print("\n[1] DATASET LOADING")
print("-" * 75)
print("Dataset shape:", df.shape)

required_columns = [
    "Sample_ID",
    "Aging_Type",
    "Aging_Time_hr",
    "Diffusion_Coefficient",
    "Tensile_Strength_MPa",
    "Elongation_at_Break_percent",
    "Youngs_Modulus_MPa",
    "Contact_Angle_Degree",
    "Dielectric_Constant",
    "Dielectric_Loss",
    "Breakdown_Strength_kVmm",
    "Trap_Depth_eV",
    "Trap_Density",
    "LIBS_Carbon_Intensity",
    "LIBS_Oxygen_Intensity",
    "LIBS_Sodium_Intensity",
    "LIBS_Magnesium_Intensity",
    "Degradation_Index",
    "Remaining_Useful_Life_hr",
    "Failure_Class"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {missing_columns}"
    )

feature_groups = {
    "Mechanical": [
        "Tensile_Strength_MPa",
        "Elongation_at_Break_percent",
        "Youngs_Modulus_MPa"
    ],

    "Physicochemical": [
        "Diffusion_Coefficient",
        "Contact_Angle_Degree"
    ],

    "Electrical": [
        "Dielectric_Constant",
        "Dielectric_Loss",
        "Breakdown_Strength_kVmm"
    ],

    "Charge_Trap": [
        "Trap_Depth_eV",
        "Trap_Density"
    ],

    "Spectroscopic_LIBS": [
        "LIBS_Carbon_Intensity",
        "LIBS_Oxygen_Intensity",
        "LIBS_Sodium_Intensity",
        "LIBS_Magnesium_Intensity"
    ]
}

feature_columns = [
    col
    for group_columns in feature_groups.values()
    for col in group_columns
]

print("\n[2] MULTIMODAL FEATURE GROUPS")
print("-" * 75)

for group_name, columns in feature_groups.items():
    print(f"{group_name}: {len(columns)} features")

print("Total measurement features:", len(feature_columns))

manifest_rows = []

for group_name, columns in feature_groups.items():
    for col in columns:
        manifest_rows.append({
            "Feature_Group": group_name,
            "Feature_Name": col,
            "Data_Type": str(df[col].dtype),
            "Unique_Values": int(df[col].nunique()),
            "Missing_Values": int(df[col].isnull().sum())
        })

feature_manifest = pd.DataFrame(manifest_rows)

feature_manifest.to_csv(
    feature_manifest_path,
    index=False
)

print("\nFeature manifest saved:", feature_manifest_path)

expected_times = [0, 168, 336, 720, 1440]

print("\n[3] AGING-TIME VALIDATION")
print("-" * 75)

print("Expected aging times:", expected_times)
print(
    "Observed aging times:",
    sorted(df["Aging_Time_hr"].unique().tolist())
)

if set(df["Aging_Time_hr"].unique()) != set(expected_times):
    raise ValueError(
        "Observed aging times do not match the expected five time points."
    )

df = df.sort_values(
    by=["Sample_ID", "Aging_Time_hr"]
).reset_index(drop=True)

records_per_sample = df.groupby("Sample_ID").size()

print("\n[4] TEMPORAL GROUP VERIFICATION")
print("-" * 75)

print("Unique samples:", df["Sample_ID"].nunique())
print("Minimum records per sample:", records_per_sample.min())
print("Maximum records per sample:", records_per_sample.max())

if not (records_per_sample == len(expected_times)).all():
    raise ValueError(
        "Some Sample_ID groups do not contain exactly five observations."
    )

invalid_samples = []

for sample_id, group in df.groupby("Sample_ID", sort=False):
    observed = group["Aging_Time_hr"].tolist()

    if observed != expected_times:
        invalid_samples.append(sample_id)

if invalid_samples:
    raise ValueError(
        f"Invalid temporal sequences detected for samples: "
        f"{invalid_samples[:10]}"
    )

print("Every Sample_ID contains the expected five time points.")

aging_type_counts = df.groupby("Sample_ID")["Aging_Type"].nunique()

mixed_aging_samples = aging_type_counts[
    aging_type_counts > 1
].index.tolist()

print("\n[5] AGING-TYPE CONSISTENCY")
print("-" * 75)

print("Samples with multiple aging types:", len(mixed_aging_samples))

if mixed_aging_samples:
    raise ValueError(
        "Some Sample_ID groups contain multiple aging types. "
        "Check sample grouping before sequence construction."
    )

X_sequences = []
sequence_ids = []
sequence_aging_types = []

target_rows = []

for sample_id, group in df.groupby("Sample_ID", sort=True):

    group = group.sort_values("Aging_Time_hr")

    X_sample = group[feature_columns].to_numpy(dtype=np.float64)

    if X_sample.shape != (5, 14):
        raise ValueError(
            f"Unexpected sequence shape for Sample_ID={sample_id}: "
            f"{X_sample.shape}"
        )

    X_sequences.append(X_sample)
    sequence_ids.append(sample_id)
    sequence_aging_types.append(group["Aging_Type"].iloc[0])

    final_row = group.iloc[-1]

    target_rows.append({
        "Sample_ID": sample_id,
        "Aging_Type": final_row["Aging_Type"],
        "Final_Aging_Time_hr": final_row["Aging_Time_hr"],
        "Degradation_Index": final_row["Degradation_Index"],
        "Remaining_Useful_Life_hr":
            final_row["Remaining_Useful_Life_hr"],
        "Failure_Class": final_row["Failure_Class"]
    })

X_sequences = np.asarray(X_sequences, dtype=np.float64)

targets = pd.DataFrame(target_rows)

print("\n[6] TEMPORAL SEQUENCE CONSTRUCTION")
print("-" * 75)

print("Sequence array shape:", X_sequences.shape)
print("Number of sequences:", X_sequences.shape[0])
print("Time steps per sequence:", X_sequences.shape[1])
print("Features per time step:", X_sequences.shape[2])

failure_encoder = LabelEncoder()

targets["Failure_Class_Encoded"] = failure_encoder.fit_transform(
    targets["Failure_Class"].astype(str)
)

print("\n[7] FAILURE-CLASS ENCODING")
print("-" * 75)

for class_name, class_code in zip(
    failure_encoder.classes_,
    failure_encoder.transform(failure_encoder.classes_)
):
    print(f"{class_name}: {class_code}")

class_mapping = pd.DataFrame({
    "Failure_Class": failure_encoder.classes_,
    "Encoded_Label": range(len(failure_encoder.classes_))
})

class_mapping.to_csv(
    os.path.join(output_folder, "step3_failure_class_mapping.csv"),
    index=False
)

np.savez_compressed(
    sequence_path,
    X_sequences=X_sequences,
    sequence_ids=np.asarray(sequence_ids),
    feature_names=np.asarray(feature_columns),
    expected_times=np.asarray(expected_times)
)

targets.to_csv(target_path, index=False)

print("\n[8] OUTPUT FILES")
print("-" * 75)
print("Temporal sequences:", sequence_path)
print("Target values:", target_path)

sequence_info = (
    df.groupby("Sample_ID")
      .agg(
          Record_Count=("Aging_Time_hr", "size"),
          Aging_Type=("Aging_Type", "first"),
          First_Aging_Time_hr=("Aging_Time_hr", "min"),
          Last_Aging_Time_hr=("Aging_Time_hr", "max"),
          Unique_Aging_Times=("Aging_Time_hr", "nunique")
      )
      .reset_index()
)

sequence_info.to_csv(
    sequence_info_path,
    index=False
)

print("Sequence verification report:", sequence_info_path)

print("\n" + "=" * 75)
print("STEP 3 COMPLETED")
print("=" * 75)

print("Original records:", len(df))
print("Unique sample sequences:", len(sequence_ids))
print("Time steps per sequence:", X_sequences.shape[1])
print("Features per time step:", X_sequences.shape[2])
print("Input shape:", X_sequences.shape)

print("\nTarget columns:")
print(targets.columns.tolist())

print("\nFinal failure-class distribution:")
print(targets["Failure_Class"].value_counts().to_string())

print("\nGenerated files:")
print("1. step3_XLPE_temporal_sequences.npz")
print("2. step3_XLPE_targets.csv")
print("3. step3_sequence_verification.csv")
print("4. step3_feature_manifest.csv")
print("5. step3_failure_class_mapping.csv")

STEP 3: MULTIMODAL FEATURE CONSTRUCTION
AND TEMPORAL SEQUENCE PREPARATION

[1] DATASET LOADING
---------------------------------------------------------------------------
Dataset shape: (1500, 20)

[2] MULTIMODAL FEATURE GROUPS
---------------------------------------------------------------------------
Mechanical: 3 features
Physicochemical: 2 features
Electrical: 3 features
Charge_Trap: 2 features
Spectroscopic_LIBS: 4 features
Total measurement features: 14

Feature manifest saved: F:\.0 Work\3931\step3_feature_manifest.csv

[3] AGING-TIME VALIDATION
---------------------------------------------------------------------------
Expected aging times: [0, 168, 336, 720, 1440]
Observed aging times: [0, 168, 336, 720, 1440]

[4] TEMPORAL GROUP VERIFICATION
---------------------------------------------------------------------------
Unique samples: 300
Minimum records per sample: 5
Maximum records per sample: 5
Every Sample_ID contains the expected five time points.

[5] AGING-TYPE CONSISTENC

# DATA PARTITIONING AND LEAKAGE-FREE NORMALIZATION

In [4]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

BASE_DIR = r"F:\.0 Work\3931"

SEQUENCE_FILE = os.path.join(
    BASE_DIR, "step3_XLPE_temporal_sequences.npz"
)

TARGET_FILE = os.path.join(
    BASE_DIR, "step3_XLPE_targets.csv"
)

OUTPUT_DIR = os.path.join(BASE_DIR, "Step4_Prepared_Data")
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42

INPUT_TIME_STEPS = 4

print("=" * 75)
print("STEP 4: DATA PARTITIONING AND NORMALIZATION")
print("=" * 75)

if not os.path.exists(SEQUENCE_FILE):
    raise FileNotFoundError(f"Sequence file not found: {SEQUENCE_FILE}")

if not os.path.exists(TARGET_FILE):
    raise FileNotFoundError(f"Target file not found: {TARGET_FILE}")

sequence_data = np.load(SEQUENCE_FILE, allow_pickle=False)
X_all = sequence_data["X_sequences"]

targets = pd.read_csv(TARGET_FILE)

print("\n[1] DATA LOADING")
print("-" * 75)
print("Full sequence shape:", X_all.shape)
print("Target table shape:", targets.shape)

required_targets = [
    "Sample_ID",
    "Aging_Type",
    "Final_Aging_Time_hr",
    "Degradation_Index",
    "Remaining_Useful_Life_hr",
    "Failure_Class"
]

missing_columns = [
    col for col in required_targets if col not in targets.columns
]

if missing_columns:
    raise ValueError(f"Missing target columns: {missing_columns}")

if X_all.ndim != 3:
    raise ValueError("Expected sequence array with shape (samples, times, features).")

if X_all.shape[0] != len(targets):
    raise ValueError(
        "Sequence count does not match target-table row count."
    )

if X_all.shape[1] < INPUT_TIME_STEPS:
    raise ValueError("Not enough time steps for the selected input window.")

if targets["Sample_ID"].duplicated().any():
    raise ValueError(
        "Target table contains duplicate Sample_ID values. "
        "Resolve sample-to-target alignment before proceeding."
    )

if targets["Sample_ID"].isna().any():
    raise ValueError("Sample_ID contains missing values.")

if targets["Failure_Class"].isna().any():
    raise ValueError("Failure_Class contains missing values.")

if targets["Aging_Type"].isna().any():
    raise ValueError("Aging_Type contains missing values.")

if not np.isfinite(X_all).all():
    raise ValueError(
        "Sequence data contains NaN or infinite values. "
        "Inspect the source data before splitting."
    )

print("Input and target validation passed.")

cleaned_file = os.path.join(BASE_DIR, "XLPE_Aging_Cleaned.csv")

if os.path.exists(cleaned_file):
    cleaned_df = pd.read_csv(cleaned_file)

    sample_order = (
        cleaned_df[["Sample_ID"]]
        .drop_duplicates()
        ["Sample_ID"]
        .tolist()
    )

    target_order = targets["Sample_ID"].tolist()

    if sample_order != target_order:
        raise ValueError(
            "Sample_ID ordering differs between the cleaned dataset "
            "and Step 3 target table. Rebuild aligned sequences and targets "
            "before continuing."
        )

    print("Sample alignment verified against the cleaned dataset.")
else:
    print(
        "WARNING: Cleaned source file not found. "
        "Alignment is assumed from Step 3 output ordering."
    )

X = X_all[:, :INPUT_TIME_STEPS, :].astype(np.float64)

sample_ids = targets["Sample_ID"].to_numpy()
aging_types = targets["Aging_Type"].astype(str).to_numpy()

y_di = pd.to_numeric(
    targets["Degradation_Index"], errors="coerce"
).to_numpy(dtype=float)

y_rul = pd.to_numeric(
    targets["Remaining_Useful_Life_hr"], errors="coerce"
).to_numpy(dtype=float)

y_class = targets["Failure_Class"].astype(str).to_numpy()

if not np.isfinite(y_di).all():
    raise ValueError("Degradation_Index contains missing or non-finite values.")

if not np.isfinite(y_rul).all():
    raise ValueError(
        "Remaining_Useful_Life_hr contains missing or non-finite values."
    )

print("\n[2] EARLY-PREDICTION INPUT PREPARATION")
print("-" * 75)
print("Input time steps:", INPUT_TIME_STEPS)
print("Input sequence shape:", X.shape)
print("Target time:", targets["Final_Aging_Time_hr"].unique())

print(
    "Prediction targets: Degradation_Index, "
    "Remaining_Useful_Life_hr, Failure_Class"
)

indices = np.arange(len(targets))

class_counts = pd.Series(y_class).value_counts()

if class_counts.min() < 3:
    raise ValueError(
        "At least one failure class has fewer than three samples; "
        "a reliable stratified three-way split is not feasible."
    )

idx_train, idx_temp = train_test_split(
    indices,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y_class
)

idx_val, idx_test = train_test_split(
    idx_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_class[idx_temp]
)

train_ids = set(sample_ids[idx_train])
val_ids = set(sample_ids[idx_val])
test_ids = set(sample_ids[idx_test])

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

print("\n[3] DATA PARTITIONING")
print("-" * 75)
print(f"Training samples:   {len(idx_train)} ({len(idx_train)/len(indices):.1%})")
print(f"Validation samples: {len(idx_val)} ({len(idx_val)/len(indices):.1%})")
print(f"Testing samples:    {len(idx_test)} ({len(idx_test)/len(indices):.1%})")

print("\nTraining class distribution:")
print(pd.Series(y_class[idx_train]).value_counts())

print("\nValidation class distribution:")
print(pd.Series(y_class[idx_val]).value_counts())

print("\nTesting class distribution:")
print(pd.Series(y_class[idx_test]).value_counts())

n_train, n_steps, n_features = X[idx_train].shape

X_train_2d = X[idx_train].reshape(-1, n_features)

scaler = StandardScaler()
scaler.fit(X_train_2d)

def transform_sequences(X_part, fitted_scaler):
    """
    Transform temporal sequences using a scaler fitted only on training data.
    """
    n_samples, n_steps, n_features = X_part.shape

    transformed = fitted_scaler.transform(
        X_part.reshape(-1, n_features)
    )

    return transformed.reshape(
        n_samples, n_steps, n_features
    ).astype(np.float32)

X_train = transform_sequences(X[idx_train], scaler)
X_val = transform_sequences(X[idx_val], scaler)
X_test = transform_sequences(X[idx_test], scaler)

y_di_train = y_di[idx_train].astype(np.float32)
y_di_val = y_di[idx_val].astype(np.float32)
y_di_test = y_di[idx_test].astype(np.float32)

y_rul_train = y_rul[idx_train].astype(np.float32)
y_rul_val = y_rul[idx_val].astype(np.float32)
y_rul_test = y_rul[idx_test].astype(np.float32)

class_names = sorted(np.unique(y_class).tolist())
class_to_index = {
    class_name: index
    for index, class_name in enumerate(class_names)
}

y_class_encoded = np.array(
    [class_to_index[c] for c in y_class],
    dtype=np.int64
)

y_class_train = y_class_encoded[idx_train]
y_class_val = y_class_encoded[idx_val]
y_class_test = y_class_encoded[idx_test]

np.savez_compressed(
    os.path.join(OUTPUT_DIR, "step4_train.npz"),
    X=X_train,
    y_degradation_index=y_di_train,
    y_remaining_useful_life=y_rul_train,
    y_failure_class=y_class_train,
    sample_ids=sample_ids[idx_train]
)

np.savez_compressed(
    os.path.join(OUTPUT_DIR, "step4_validation.npz"),
    X=X_val,
    y_degradation_index=y_di_val,
    y_remaining_useful_life=y_rul_val,
    y_failure_class=y_class_val,
    sample_ids=sample_ids[idx_val]
)

np.savez_compressed(
    os.path.join(OUTPUT_DIR, "step4_test.npz"),
    X=X_test,
    y_degradation_index=y_di_test,
    y_remaining_useful_life=y_rul_test,
    y_failure_class=y_class_test,
    sample_ids=sample_ids[idx_test]
)

np.savez_compressed(
    os.path.join(OUTPUT_DIR, "step4_scaler_parameters.npz"),
    mean=scaler.mean_,
    scale=scaler.scale_,
    var=scaler.var_
)

pd.DataFrame({
    "Failure_Class": class_names,
    "Encoded_Label": [class_to_index[c] for c in class_names]
}).to_csv(
    os.path.join(OUTPUT_DIR, "step4_failure_class_mapping.csv"),
    index=False
)

split_map = {}

for i in idx_train:
    split_map[sample_ids[i]] = "Train"

for i in idx_val:
    split_map[sample_ids[i]] = "Validation"

for i in idx_test:
    split_map[sample_ids[i]] = "Test"

split_report = targets[
    ["Sample_ID", "Aging_Type", "Failure_Class"]
].copy()

split_report["Dataset_Split"] = split_report["Sample_ID"].map(split_map)

split_report.to_csv(
    os.path.join(OUTPUT_DIR, "step4_sample_split_report.csv"),
    index=False
)

verification = pd.DataFrame({
    "Partition": ["Train", "Validation", "Test"],
    "Samples": [len(idx_train), len(idx_val), len(idx_test)],
    "Input_Time_Steps": [INPUT_TIME_STEPS] * 3,
    "Features_Per_Time_Step": [n_features] * 3,
    "Input_Shape": [
        str(X_train.shape),
        str(X_val.shape),
        str(X_test.shape)
    ]
})

verification.to_csv(
    os.path.join(OUTPUT_DIR, "step4_verification_report.csv"),
    index=False
)

print("\n[4] FINAL SHAPES")
print("-" * 75)
print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)

print("\n[5] TARGET SHAPES")
print("-" * 75)
print("Degradation Index:", y_di_train.shape, y_di_val.shape, y_di_test.shape)
print("Remaining Useful Life:", y_rul_train.shape, y_rul_val.shape, y_rul_test.shape)
print("Failure Class:", y_class_train.shape, y_class_val.shape, y_class_test.shape)

print("\n[6] LEAKAGE CHECKS")
print("-" * 75)
print("Train/validation overlap:", len(train_ids & val_ids))
print("Train/test overlap:", len(train_ids & test_ids))
print("Validation/test overlap:", len(val_ids & test_ids))

print("Scaler fitted on training data only: Yes")
print("Input time points used: First four")
print("Final time point used as target: 1440 hours")

print("\n[7] OUTPUT DIRECTORY")
print("-" * 75)
print(OUTPUT_DIR)

print("\nGenerated files:")
for filename in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", filename)

print("\n" + "=" * 75)
print("STEP 4 COMPLETED SUCCESSFULLY")
print("=" * 75)

STEP 4: DATA PARTITIONING AND NORMALIZATION

[1] DATA LOADING
---------------------------------------------------------------------------
Full sequence shape: (300, 5, 14)
Target table shape: (300, 7)
Input and target validation passed.
Sample alignment verified against the cleaned dataset.

[2] EARLY-PREDICTION INPUT PREPARATION
---------------------------------------------------------------------------
Input time steps: 4
Input sequence shape: (300, 4, 14)
Target time: [1440]
Prediction targets: Degradation_Index, Remaining_Useful_Life_hr, Failure_Class

[3] DATA PARTITIONING
---------------------------------------------------------------------------
Training samples:   210 (70.0%)
Validation samples: 45 (15.0%)
Testing samples:    45 (15.0%)

Training class distribution:
Mild        171
Moderate     39
Name: count, dtype: int64

Validation class distribution:
Mild        37
Moderate     8
Name: count, dtype: int64

Testing class distribution:
Mild        37
Moderate     8
Name: coun

# SSL-STACKED LSTM MODEL DEVELOPMENT AND TRAINING

In [6]:
import os
import json
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dense,
    Dropout,
    BatchNormalization
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

warnings.filterwarnings("ignore")

BASE_DIR = r"F:\.0 Work\3931"

DATA_DIR = os.path.join(
    BASE_DIR,
    "Step4_Prepared_Data"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "Step5_Model_Results"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42
EPOCHS = 150
BATCH_SIZE = 16
LEARNING_RATE = 0.001

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("=" * 80)
print("STEP 5: STACKED LSTM MODEL DEVELOPMENT AND TRAINING")
print("=" * 80)

print("TensorFlow version:", tf.__version__)
print("Output directory:", OUTPUT_DIR)

def load_partition(filename):

    filepath = os.path.join(DATA_DIR, filename)

    if not os.path.isfile(filepath):
        raise FileNotFoundError(
            f"Required Step 4 file was not found:\n{filepath}"
        )

    return np.load(filepath, allow_pickle=False)


train_data = load_partition("step4_train.npz")
val_data = load_partition("step4_validation.npz")
test_data = load_partition("step4_test.npz")

X_train = train_data["X"].astype(np.float32)
X_val = val_data["X"].astype(np.float32)
X_test = test_data["X"].astype(np.float32)

y_di_train = train_data[
    "y_degradation_index"
].astype(np.float32).reshape(-1)

y_di_val = val_data[
    "y_degradation_index"
].astype(np.float32).reshape(-1)

y_di_test = test_data[
    "y_degradation_index"
].astype(np.float32).reshape(-1)

y_rul_train = train_data[
    "y_remaining_useful_life"
].astype(np.float32).reshape(-1)

y_rul_val = val_data[
    "y_remaining_useful_life"
].astype(np.float32).reshape(-1)

y_rul_test = test_data[
    "y_remaining_useful_life"
].astype(np.float32).reshape(-1)

y_class_train = train_data[
    "y_failure_class"
].astype(np.int32).reshape(-1)

y_class_val = val_data[
    "y_failure_class"
].astype(np.int32).reshape(-1)

y_class_test = test_data[
    "y_failure_class"
].astype(np.int32).reshape(-1)

train_ids = train_data["sample_ids"]
val_ids = val_data["sample_ids"]
test_ids = test_data["sample_ids"]

print("\n[1] DATA LOADING")
print("-" * 80)

print("Training input:", X_train.shape)
print("Validation input:", X_val.shape)
print("Testing input:", X_test.shape)

if X_train.ndim != 3:
    raise ValueError("Expected three-dimensional training input.")

if X_train.shape[1:] != X_val.shape[1:]:
    raise ValueError("Training and validation input shapes do not match.")

if X_train.shape[1:] != X_test.shape[1:]:
    raise ValueError("Training and testing input shapes do not match.")

if X_train.shape[1:] != (4, 14):
    raise ValueError(
        f"Expected input dimensions (4, 14), got {X_train.shape[1:]}."
    )

for name, array in [
    ("X_train", X_train),
    ("X_val", X_val),
    ("X_test", X_test),
    ("DI_train", y_di_train),
    ("DI_val", y_di_val),
    ("DI_test", y_di_test),
    ("RUL_train", y_rul_train),
    ("RUL_val", y_rul_val),
    ("RUL_test", y_rul_test)
]:
    if not np.isfinite(array).all():
        raise ValueError(
            f"{name} contains NaN or infinite values."
        )

if not (
    len(X_train) == len(y_di_train)
    == len(y_rul_train)
    == len(y_class_train)
):
    raise ValueError("Training input and target lengths do not match.")

if not (
    len(X_val) == len(y_di_val)
    == len(y_rul_val)
    == len(y_class_val)
):
    raise ValueError("Validation input and target lengths do not match.")

if not (
    len(X_test) == len(y_di_test)
    == len(y_rul_test)
    == len(y_class_test)
):
    raise ValueError("Test input and target lengths do not match.")

train_id_set = set(train_ids.tolist())
val_id_set = set(val_ids.tolist())
test_id_set = set(test_ids.tolist())

if train_id_set & val_id_set:
    raise ValueError("Training-validation sample overlap detected.")

if train_id_set & test_id_set:
    raise ValueError("Training-test sample overlap detected.")

if val_id_set & test_id_set:
    raise ValueError("Validation-test sample overlap detected.")

print("\n[2] VALIDATION")
print("-" * 80)
print("Input and target dimensions verified.")
print("No non-finite values detected.")
print("Sample-level leakage checks passed.")

mapping_path = os.path.join(
    DATA_DIR,
    "step4_failure_class_mapping.csv"
)

if not os.path.isfile(mapping_path):
    raise FileNotFoundError(
        f"Class mapping file not found: {mapping_path}"
    )

class_mapping = pd.read_csv(mapping_path)

required_mapping_columns = [
    "Failure_Class",
    "Encoded_Label"
]

if not all(
    col in class_mapping.columns
    for col in required_mapping_columns
):
    raise ValueError(
        "Class mapping file must contain Failure_Class and Encoded_Label."
    )

class_mapping = class_mapping.sort_values(
    "Encoded_Label"
).reset_index(drop=True)

class_names = class_mapping[
    "Failure_Class"
].astype(str).tolist()

encoded_labels = class_mapping[
    "Encoded_Label"
].to_numpy(dtype=int)

if not np.array_equal(
    encoded_labels,
    np.arange(len(class_names))
):
    raise ValueError(
        "Class labels must be contiguous integers starting from zero."
    )

num_classes = len(class_names)

if num_classes < 2:
    raise ValueError("At least two classes are required.")

for partition_name, labels in [
    ("Training", y_class_train),
    ("Validation", y_class_val),
    ("Testing", y_class_test)
]:

    if not np.isin(labels, encoded_labels).all():
        raise ValueError(
            f"{partition_name} labels do not match the class mapping."
        )

print("\n[3] FAILURE CLASSES")
print("-" * 80)
print(class_mapping.to_string(index=False))

for i, class_name in enumerate(class_names):
    print(
        f"{class_name}: "
        f"training={(y_class_train == i).sum()}, "
        f"validation={(y_class_val == i).sum()}, "
        f"testing={(y_class_test == i).sum()}"
    )

di_scaler = StandardScaler()

rul_scaler = StandardScaler()

y_di_train_scaled = di_scaler.fit_transform(
    y_di_train.reshape(-1, 1)
).astype(np.float32)

y_di_val_scaled = di_scaler.transform(
    y_di_val.reshape(-1, 1)
).astype(np.float32)

y_di_test_scaled = di_scaler.transform(
    y_di_test.reshape(-1, 1)
).astype(np.float32)


y_rul_train_scaled = rul_scaler.fit_transform(
    y_rul_train.reshape(-1, 1)
).astype(np.float32)

y_rul_val_scaled = rul_scaler.transform(
    y_rul_val.reshape(-1, 1)
).astype(np.float32)

y_rul_test_scaled = rul_scaler.transform(
    y_rul_test.reshape(-1, 1)
).astype(np.float32)

print("\n[4] TARGET SCALING")
print("-" * 80)
print("DI target scaler fitted on training data only.")
print("RUL target scaler fitted on training data only.")

class_counts = np.bincount(
    y_class_train,
    minlength=num_classes
)

if np.any(class_counts == 0):
    raise ValueError(
        "At least one class is absent from the training partition."
    )

class_weights = (
    len(y_class_train)
    / (num_classes * class_counts)
).astype(np.float32)

sample_weights_class = class_weights[
    y_class_train
].astype(np.float32)

print("\n[5] CLASS IMBALANCE HANDLING")
print("-" * 80)

for i, class_name in enumerate(class_names):
    print(
        f"{class_name}: "
        f"count={class_counts[i]}, "
        f"weight={class_weights[i]:.4f}"
    )

time_steps = X_train.shape[1]
num_features = X_train.shape[2]

inputs = Input(
    shape=(time_steps, num_features),
    name="temporal_input"
)

x = LSTM(
    64,
    return_sequences=True,
    name="lstm_layer_1"
)(inputs)

x = BatchNormalization(
    name="batch_norm_1"
)(x)

x = Dropout(
    0.25,
    name="dropout_1"
)(x)

x = LSTM(
    32,
    return_sequences=False,
    name="lstm_layer_2"
)(x)

x = BatchNormalization(
    name="batch_norm_2"
)(x)

x = Dropout(
    0.25,
    name="dropout_2"
)(x)

shared = Dense(
    32,
    activation="relu",
    name="shared_dense"
)(x)

di_branch = Dense(
    16,
    activation="relu",
    name="di_hidden"
)(shared)

di_output = Dense(
    1,
    activation="linear",
    name="degradation_index"
)(di_branch)

rul_branch = Dense(
    16,
    activation="relu",
    name="rul_hidden"
)(shared)

rul_output = Dense(
    1,
    activation="linear",
    name="remaining_useful_life"
)(rul_branch)

class_branch = Dense(
    16,
    activation="relu",
    name="class_hidden"
)(shared)

class_output = Dense(
    num_classes,
    activation="softmax",
    name="failure_class"
)(class_branch)

model = Model(
    inputs=inputs,
    outputs={
        "degradation_index": di_output,
        "remaining_useful_life": rul_output,
        "failure_class": class_output
    },
    name="Stacked_LSTM_Multitask_Model"
)

print("\n[6] MODEL OUTPUT VERIFICATION")
print("-" * 80)
print("Output names:", model.output_names)

assert set(model.output_names) == {
    "degradation_index",
    "remaining_useful_life",
    "failure_class"
}

print("Named model outputs verified.")

model.compile(
    optimizer=Adam(
        learning_rate=LEARNING_RATE,
        clipnorm=1.0
    ),

    loss={
        "degradation_index": "mse",
        "remaining_useful_life": "mse",
        "failure_class": "sparse_categorical_crossentropy"
    },

    loss_weights={
        "degradation_index": 1.0,
        "remaining_useful_life": 1.0,
        "failure_class": 1.0
    },

    metrics={
        "degradation_index": ["mae"],
        "remaining_useful_life": ["mae"],
        "failure_class": ["accuracy"]
    }
)

model.summary()

architecture_path = os.path.join(
    OUTPUT_DIR,
    "model_architecture.txt"
)

with open(
    architecture_path,
    "w",
    encoding="utf-8"
) as f:
    model.summary(
        print_fn=lambda line: f.write(line + "\n")
    )

best_model_path = os.path.join(
    OUTPUT_DIR,
    "best_stacked_lstm_model.keras"
)

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=20,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=8,
        min_lr=1e-6,
        verbose=1
    ),

    ModelCheckpoint(
        filepath=best_model_path,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]

print("\n[7] MODEL TRAINING")
print("-" * 80)

train_targets = {
    "degradation_index": y_di_train_scaled,
    "remaining_useful_life": y_rul_train_scaled,
    "failure_class": y_class_train
}

validation_targets = {
    "degradation_index": y_di_val_scaled,
    "remaining_useful_life": y_rul_val_scaled,
    "failure_class": y_class_val
}

train_sample_weights = {
    "degradation_index": np.ones(
        len(X_train), dtype=np.float32
    ),

    "remaining_useful_life": np.ones(
        len(X_train), dtype=np.float32
    ),

    "failure_class": sample_weights_class
}

assert len(train_targets["degradation_index"]) == len(X_train)
assert len(train_targets["remaining_useful_life"]) == len(X_train)
assert len(train_targets["failure_class"]) == len(X_train)

assert len(validation_targets["degradation_index"]) == len(X_val)
assert len(validation_targets["remaining_useful_life"]) == len(X_val)
assert len(validation_targets["failure_class"]) == len(X_val)

history = model.fit(
    X_train,
    train_targets,

    validation_data=(
        X_val,
        validation_targets
    ),

    sample_weight=train_sample_weights,

    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

print("\nModel training completed successfully.")

history_df = pd.DataFrame(history.history)

history_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "step5_training_history.csv"
    ),
    index=False
)

plt.figure(figsize=(10, 6))

plt.plot(
    history.history["loss"],
    label="Training Loss",
    linewidth=2
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss",
    linewidth=2
)

plt.xlabel("Epoch")
plt.ylabel("Total Loss")
plt.title("Stacked LSTM Training and Validation Loss")

plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "step5_total_loss_curve.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()

accuracy_key = next(
    (
        key for key in [
            "failure_class_accuracy",
            "failure_class_sparse_categorical_accuracy"
        ]
        if key in history.history
    ),
    None
)

if accuracy_key is None:
    raise KeyError(
        "Classification accuracy history not found. "
        f"Available history keys: {list(history.history.keys())}"
    )

validation_accuracy_key = "val_" + accuracy_key

plt.figure(figsize=(10, 6))

plt.plot(
    history.history[accuracy_key],
    label="Training Accuracy",
    linewidth=2
)

plt.plot(
    history.history[validation_accuracy_key],
    label="Validation Accuracy",
    linewidth=2
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Stacked LSTM Failure Classification Accuracy")

plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "step5_accuracy_curve.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()

for task_name in [
    "degradation_index",
    "remaining_useful_life"
]:

    train_key = task_name + "_loss"
    val_key = "val_" + task_name + "_loss"

    if train_key not in history.history:
        continue

    plt.figure(figsize=(10, 6))

    plt.plot(
        history.history[train_key],
        label="Training Loss",
        linewidth=2
    )

    plt.plot(
        history.history[val_key],
        label="Validation Loss",
        linewidth=2
    )

    plt.xlabel("Epoch")
    plt.ylabel("Mean Squared Error")

    plt.title(
        task_name.replace("_", " ").title()
        + " Training and Validation Loss"
    )

    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()

    plt.savefig(
        os.path.join(
            OUTPUT_DIR,
            task_name + "_loss_curve.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

print("\n[8] TEST PREDICTION")
print("-" * 80)

predictions = model.predict(
    X_test,
    batch_size=BATCH_SIZE,
    verbose=1
)

if not isinstance(predictions, dict):
    raise TypeError(
        "Expected dictionary predictions from the named-output model. "
        f"Received: {type(predictions)}"
    )

pred_di_scaled = np.asarray(
    predictions["degradation_index"]
).reshape(-1, 1)

pred_rul_scaled = np.asarray(
    predictions["remaining_useful_life"]
).reshape(-1, 1)

pred_class_probabilities = np.asarray(
    predictions["failure_class"]
)

pred_di = di_scaler.inverse_transform(
    pred_di_scaled
).reshape(-1)

pred_rul = rul_scaler.inverse_transform(
    pred_rul_scaled
).reshape(-1)

pred_class = np.argmax(
    pred_class_probabilities,
    axis=1
)

print("Predictions generated successfully.")

def calculate_regression_metrics(y_true, y_pred):

    return {
        "MAE": float(
            mean_absolute_error(y_true, y_pred)
        ),

        "RMSE": float(
            np.sqrt(
                mean_squared_error(y_true, y_pred)
            )
        ),

        "R2": float(
            r2_score(y_true, y_pred)
        )
    }


di_metrics = calculate_regression_metrics(
    y_di_test,
    pred_di
)

rul_metrics = calculate_regression_metrics(
    y_rul_test,
    pred_rul
)

class_metrics = {
    "Accuracy": float(
        accuracy_score(y_class_test, pred_class)
    ),

    "Precision_Macro": float(
        precision_score(
            y_class_test,
            pred_class,
            labels=np.arange(num_classes),
            average="macro",
            zero_division=0
        )
    ),

    "Recall_Macro": float(
        recall_score(
            y_class_test,
            pred_class,
            labels=np.arange(num_classes),
            average="macro",
            zero_division=0
        )
    ),

    "F1_Macro": float(
        f1_score(
            y_class_test,
            pred_class,
            labels=np.arange(num_classes),
            average="macro",
            zero_division=0
        )
    ),

    "F1_Weighted": float(
        f1_score(
            y_class_test,
            pred_class,
            labels=np.arange(num_classes),
            average="weighted",
            zero_division=0
        )
    )
}

metrics_df = pd.DataFrame([
    {
        "Task": "Degradation Index",
        **di_metrics
    },

    {
        "Task": "Remaining Useful Life",
        **rul_metrics
    },

    {
        "Task": "Failure Classification",
        **class_metrics
    }
])

metrics_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "step5_test_metrics.csv"
    ),
    index=False
)

print("\n[9] TEST METRICS")
print("-" * 80)

print("\nDegradation Index:")
for metric, value in di_metrics.items():
    print(f"{metric}: {value:.6f}")

print("\nRemaining Useful Life:")
for metric, value in rul_metrics.items():
    print(f"{metric}: {value:.6f}")

print("\nFailure Classification:")
for metric, value in class_metrics.items():
    print(f"{metric}: {value:.6f}")

report = classification_report(
    y_class_test,
    pred_class,
    labels=np.arange(num_classes),
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()

report_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "step5_classification_report.csv"
    )
)

print("\nClassification report:")
print(
    classification_report(
        y_class_test,
        pred_class,
        labels=np.arange(num_classes),
        target_names=class_names,
        zero_division=0
    )
)

cm = confusion_matrix(
    y_class_test,
    pred_class,
    labels=np.arange(num_classes)
)

cm_df = pd.DataFrame(
    cm,
    index=[
        "Actual_" + name for name in class_names
    ],
    columns=[
        "Predicted_" + name for name in class_names
    ]
)

cm_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "step5_confusion_matrix.csv"
    )
)

plt.figure(figsize=(7, 6))

plt.imshow(
    cm,
    interpolation="nearest",
    aspect="auto"
)

plt.title("Stacked LSTM Failure Classification Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")

plt.xticks(
    np.arange(num_classes),
    class_names,
    rotation=30,
    ha="right"
)

plt.yticks(
    np.arange(num_classes),
    class_names
)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(
            j,
            i,
            str(cm[i, j]),
            ha="center",
            va="center"
        )

plt.colorbar(label="Number of Samples")
plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "step5_confusion_matrix.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()

prediction_df = pd.DataFrame({
    "Sample_ID": test_ids,

    "Actual_Degradation_Index": y_di_test,
    "Predicted_Degradation_Index": pred_di,

    "Actual_Remaining_Useful_Life_hr": y_rul_test,
    "Predicted_Remaining_Useful_Life_hr": pred_rul,

    "Actual_Failure_Class_Encoded": y_class_test,
    "Predicted_Failure_Class_Encoded": pred_class,

    "Actual_Failure_Class": [
        class_names[i] for i in y_class_test
    ],

    "Predicted_Failure_Class": [
        class_names[i] for i in pred_class
    ]
})

for class_index, class_name in enumerate(class_names):

    safe_name = (
        class_name.lower()
        .replace(" ", "_")
        .replace("/", "_")
    )

    prediction_df[
        "Probability_" + safe_name
    ] = pred_class_probabilities[:, class_index]

prediction_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "step5_test_predictions.csv"
    ),
    index=False
)

np.savez_compressed(
    os.path.join(
        OUTPUT_DIR,
        "step5_target_scalers.npz"
    ),
    di_mean=di_scaler.mean_,
    di_scale=di_scaler.scale_,
    rul_mean=rul_scaler.mean_,
    rul_scale=rul_scaler.scale_
)

configuration = {
    "model_name": "Stacked_LSTM_Multitask_Model",
    "random_seed": RANDOM_STATE,
    "input_time_steps": int(time_steps),
    "input_features": int(num_features),
    "training_samples": int(len(X_train)),
    "validation_samples": int(len(X_val)),
    "testing_samples": int(len(X_test)),
    "requested_epochs": int(EPOCHS),
    "completed_epochs": int(len(history.history["loss"])),
    "batch_size": int(BATCH_SIZE),
    "learning_rate": float(LEARNING_RATE),
    "failure_classes": class_names,
    "outputs": [
        "Degradation Index",
        "Remaining Useful Life",
        "Failure Class"
    ],
    "input_aging_times_hours": [
        0, 168, 336, 720
    ],
    "target_aging_time_hours": 1440,
    "optimizer": "Adam",
    "seven_spot_ladybird_optimization_integrated": False
}

with open(
    os.path.join(
        OUTPUT_DIR,
        "step5_model_configuration.json"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        configuration,
        f,
        indent=4
    )

print("\n[10] GENERATED OUTPUT FILES")
print("-" * 80)

for filename in sorted(os.listdir(OUTPUT_DIR)):
    print(filename)

print("\n" + "=" * 80)
print("STEP 5 COMPLETED SUCCESSFULLY")
print("=" * 80)

print("Model:", model.name)
print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))
print("Input shape:", X_train.shape[1:])
print("Number of prediction tasks: 3")
print("Best model:", best_model_path)
print("Results directory:", OUTPUT_DIR)

print("\nMeasured test results:")
print(metrics_df.to_string(index=False))

STEP 5: STACKED LSTM MODEL DEVELOPMENT AND TRAINING
TensorFlow version: 2.20.0
Output directory: F:\.0 Work\3931\Step5_Model_Results

[1] DATA LOADING
--------------------------------------------------------------------------------
Training input: (210, 4, 14)
Validation input: (45, 4, 14)
Testing input: (45, 4, 14)

[2] VALIDATION
--------------------------------------------------------------------------------
Input and target dimensions verified.
No non-finite values detected.
Sample-level leakage checks passed.

[3] FAILURE CLASSES
--------------------------------------------------------------------------------
Failure_Class  Encoded_Label
         Mild              0
     Moderate              1
Mild: training=171, validation=37, testing=37
Moderate: training=39, validation=8, testing=8

[4] TARGET SCALING
--------------------------------------------------------------------------------
DI target scaler fitted on training data only.
RUL target scaler fitted on training data only.

[

Model: "Stacked_LSTM_Multitask_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ temporal_input (InputLayer)   │ (None, 4, 14)             │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm_layer_1 (LSTM)           │ (None, 4, 64)             │          20,224 │ temporal_input[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_norm_1                  │ (None, 4, 64)             │             256 │ lstm_layer_1[0][0]         │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_1 (Dropout)           │ (None, 4, 64)             │               0 │ batch_norm_1[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm_layer_2 (LSTM)           │ (None, 32)                │          12,416 │ dropout_1[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_norm_2                  │ (None, 32)                │             128 │ lstm_layer_2[0][0]         │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_2 (Dropout)           │ (None, 32)                │               0 │ batch_norm_2[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ shared_dense (Dense)          │ (None, 32)                │           1,056 │ dropout_2[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ di_hidden (Dense)             │ (None, 16)                │             528 │ shared_dense[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ class_hidden (Dense)          │ (None, 16)                │             528 │ shared_dense[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ rul_hidden (Dense)            │ (None, 16)                │             528 │ shared_dense[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ degradation_index (Dense)     │ (None, 1)                 │              17 │ di_hidden[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ failure_class (Dense)         │ (None, 2)                 │              34 │ class_hidden[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ remaining_useful_life (Dense) │ (None, 1)                 │              17 │ rul_hidden[0][0]           │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 35,732 (139.58 KB)

 Trainable params: 35,540 (138.83 KB)

 Non-trainable params: 192 (768.00 B)


[7] MODEL TRAINING
--------------------------------------------------------------------------------
Epoch 1/150
10/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - degradation_index_loss: 1.0407 - degradation_index_mae: 0.8076 - failure_class_accuracy: 0.6879 - failure_class_loss: 0.7356 - loss: 2.9549 - remaining_useful_life_loss: 1.1786 - remaining_useful_life_mae: 0.9399 
Epoch 1: val_loss improved from None to 2.79359, saving model to F:\.0 Work\3931\Step5_Model_Results\best_stacked_lstm_model.keras
14/14 ━━━━━━━━━━━━━━━━━━━━ 14s 134ms/step - degradation_index_loss: 0.7243 - degradation_index_mae: 0.6965 - failure_class_accuracy: 0.6333 - failure_class_loss: 0.6845 - loss: 2.2809 - remaining_useful_life_loss: 0.7948 - remaining_useful_life_mae: 0.7665 - val_degradation_index_loss: 1.0127 - val_degradation_index_mae: 0.9383 - val_failure_class_accuracy: 0.4889 - val_failure_class_loss: 0.6915 - val_loss: 2.7936 - val_remaining_useful_life_loss: 1.1094 - val_remaining_useful_life_mae: 0.9853 

# CROSS-ENTROPY ERROR 

In [22]:
import os
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import log_loss

from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input,
    Dense,
    LSTM,
    Dropout,
    BatchNormalization
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings("ignore")

BASE_DIR = r"F:\.0 Work\3931"

SEQUENCE_FILE = os.path.join(
    BASE_DIR,
    "step3_XLPE_temporal_sequences.npz"
)

TARGET_FILE = os.path.join(
    BASE_DIR,
    "step3_XLPE_targets.csv"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "CE_ANN_vs_SSL_Comparison"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42
EPOCHS = 100
BATCH_SIZE = 16
LEARNING_RATE = 0.001

SPLITS = {
    "60:40": 0.60,
    "70:30": 0.70,
    "80:20": 0.80,
    "90:10": 0.90
}

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("=" * 80)
print("ANN VS SSL-STACKED LSTM: CROSS-ENTROPY ERROR COMPARISON")
print("=" * 80)

if not os.path.exists(SEQUENCE_FILE):
    raise FileNotFoundError(SEQUENCE_FILE)

if not os.path.exists(TARGET_FILE):
    raise FileNotFoundError(TARGET_FILE)

sequence_data = np.load(
    SEQUENCE_FILE,
    allow_pickle=False
)

X_all = sequence_data["X_sequences"]

targets = pd.read_csv(TARGET_FILE)

required_columns = [
    "Sample_ID",
    "Failure_Class",
    "Final_Aging_Time_hr"
]

for column in required_columns:
    if column not in targets.columns:
        raise ValueError(
            f"Required target column missing: {column}"
        )

if len(X_all) != len(targets):
    raise ValueError(
        "Sequence count and target count do not match."
    )

if targets["Sample_ID"].duplicated().any():
    raise ValueError(
        "Sample_ID values are duplicated in the target table."
    )

if not np.isfinite(X_all).all():
    raise ValueError(
        "Input sequences contain NaN or infinite values."
    )

print("\n[1] DATASET")
print("-" * 80)
print("Full sequence shape:", X_all.shape)
print("Target table shape:", targets.shape)

INPUT_TIME_STEPS = 4

X = X_all[:, :INPUT_TIME_STEPS, :].astype(np.float32)

y_text = targets["Failure_Class"].astype(str).to_numpy()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)

class_names = label_encoder.classes_
num_classes = len(class_names)

if num_classes < 2:
    raise ValueError("At least two failure classes are required.")

if num_classes != 2:
    print(
        "Warning: this code supports multiclass classification too, "
        "but the current dataset is expected to contain two classes."
    )

if pd.Series(y).value_counts().min() < 4:
    raise ValueError(
        "Too few samples in one class for a stable four-split comparison."
    )

print("\n[2] CLASSIFICATION TARGET")
print("-" * 80)
print("Input shape:", X.shape)
print("Classes:", list(class_names))
print("\nClass counts:")
print(pd.Series(y_text).value_counts())

def build_ann(input_dim, n_classes):
    """
    ANN baseline using flattened temporal measurements.
    """

    inputs = Input(
        shape=(input_dim,),
        name="ann_input"
    )

    x = Dense(
        64,
        activation="relu"
    )(inputs)

    x = BatchNormalization()(x)
    x = Dropout(0.25)(x)

    x = Dense(
        32,
        activation="relu"
    )(x)

    x = Dropout(0.20)(x)

    outputs = Dense(
        n_classes,
        activation="softmax",
        name="failure_class"
    )(x)

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name="ANN"
    )

    model.compile(
        optimizer=Adam(
            learning_rate=LEARNING_RATE
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


def build_ssl_stacked_lstm(time_steps, n_features, n_classes):
    """
    Proposed Stacked LSTM classification baseline.
    """

    inputs = Input(
        shape=(time_steps, n_features),
        name="temporal_input"
    )

    x = LSTM(
        64,
        return_sequences=True
    )(inputs)

    x = BatchNormalization()(x)
    x = Dropout(0.25)(x)

    x = LSTM(
        32,
        return_sequences=False
    )(x)

    x = BatchNormalization()(x)
    x = Dropout(0.25)(x)

    x = Dense(
        32,
        activation="relu"
    )(x)

    x = Dropout(0.20)(x)

    outputs = Dense(
        n_classes,
        activation="softmax",
        name="failure_class"
    )(x)

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name="SSL_Stacked_LSTM"
    )

    model.compile(
        optimizer=Adam(
            learning_rate=LEARNING_RATE
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

results = []
prediction_records = []
split_records = []

for split_number, (split_name, train_ratio) in enumerate(
    SPLITS.items()
):

    print("\n" + "=" * 80)
    print(f"DATA SPLIT: {split_name}")
    print("=" * 80)

    test_ratio = 1.0 - train_ratio

    indices = np.arange(len(y))

    train_idx, test_idx = train_test_split(
        indices,
        train_size=train_ratio,
        random_state=RANDOM_STATE,
        stratify=y
    )

    assert set(train_idx).isdisjoint(set(test_idx))

    X_train_seq = X[train_idx]
    X_test_seq = X[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    n_train, n_steps, n_features = X_train_seq.shape

    scaler = StandardScaler()

    scaler.fit(
        X_train_seq.reshape(-1, n_features)
    )

    def scale_sequences(data):
        n_samples, n_steps_local, n_features_local = data.shape

        transformed = scaler.transform(
            data.reshape(-1, n_features_local)
        )

        return transformed.reshape(
            n_samples,
            n_steps_local,
            n_features_local
        ).astype(np.float32)

    X_train_seq_scaled = scale_sequences(X_train_seq)
    X_test_seq_scaled = scale_sequences(X_test_seq)

    X_train_ann = X_train_seq_scaled.reshape(
        len(X_train_seq_scaled),
        -1
    )

    X_test_ann = X_test_seq_scaled.reshape(
        len(X_test_seq_scaled),
        -1
    )

    tf.keras.backend.clear_session()

    ann = build_ann(
        input_dim=X_train_ann.shape[1],
        n_classes=num_classes
    )

    ann_callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=12,
            restore_best_weights=True,
            verbose=0
        )
    ]

    inner_train_idx, inner_val_idx = train_test_split(
        np.arange(len(y_train)),
        test_size=0.15,
        random_state=RANDOM_STATE,
        stratify=y_train
    )

    ann.fit(
        X_train_ann[inner_train_idx],
        y_train[inner_train_idx],
        validation_data=(
            X_train_ann[inner_val_idx],
            y_train[inner_val_idx]
        ),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=ann_callbacks,
        verbose=0
    )

    ann_probabilities = ann.predict(
        X_test_ann,
        verbose=0
    )

    ann_ce = log_loss(
        y_test,
        ann_probabilities,
        labels=np.arange(num_classes)
    )

    tf.keras.backend.clear_session()

    ssl_model = build_ssl_stacked_lstm(
        time_steps=n_steps,
        n_features=n_features,
        n_classes=num_classes
    )

    ssl_callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=12,
            restore_best_weights=True,
            verbose=0
        )
    ]

    ssl_model.fit(
        X_train_seq_scaled[inner_train_idx],
        y_train[inner_train_idx],
        validation_data=(
            X_train_seq_scaled[inner_val_idx],
            y_train[inner_val_idx]
        ),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=ssl_callbacks,
        verbose=0
    )

    ssl_probabilities = ssl_model.predict(
        X_test_seq_scaled,
        verbose=0
    )

    ssl_ce = log_loss(
        y_test,
        ssl_probabilities,
        labels=np.arange(num_classes)
    )

    results.append({
        "Data_Split": split_name,
        "Training_Ratio": train_ratio,
        "Testing_Ratio": test_ratio,
        "Training_Samples": len(train_idx),
        "Testing_Samples": len(test_idx),
        "ANN_Cross_Entropy": ann_ce,
        "SSL_Stacked_LSTM_Cross_Entropy": ssl_ce
    })

    print(f"Training samples: {len(train_idx)}")
    print(f"Testing samples: {len(test_idx)}")
    print(f"ANN CE: {ann_ce:.6f}")
    print(f"SSL-Stacked LSTM CE: {ssl_ce:.6f}")

    for idx in train_idx:
        split_records.append({
            "Data_Split": split_name,
            "Sample_ID": targets.iloc[idx]["Sample_ID"],
            "Partition": "Training"
        })

    for idx in test_idx:
        split_records.append({
            "Data_Split": split_name,
            "Sample_ID": targets.iloc[idx]["Sample_ID"],
            "Partition": "Testing"
        })

    for local_idx, original_idx in enumerate(test_idx):

        actual_class = class_names[y_test[local_idx]]

        row = {
            "Data_Split": split_name,
            "Sample_ID": targets.iloc[original_idx]["Sample_ID"],
            "Actual_Failure_Class": actual_class,
            "ANN_Predicted_Class": class_names[
                np.argmax(ann_probabilities[local_idx])
            ],
            "SSL_Predicted_Class": class_names[
                np.argmax(ssl_probabilities[local_idx])
            ],
            "ANN_Cross_Entropy_Sample": -np.log(
                np.clip(
                    ann_probabilities[
                        local_idx, y_test[local_idx]
                    ],
                    1e-15,
                    1.0
                )
            ),
            "SSL_Cross_Entropy_Sample": -np.log(
                np.clip(
                    ssl_probabilities[
                        local_idx, y_test[local_idx]
                    ],
                    1e-15,
                    1.0
                )
            )
        }

        for class_index, class_name in enumerate(class_names):
            row[
                f"ANN_Probability_{class_name}"
            ] = ann_probabilities[local_idx, class_index]

            row[
                f"SSL_Probability_{class_name}"
            ] = ssl_probabilities[local_idx, class_index]

        prediction_records.append(row)

    del ann
    del ssl_model

results_df = pd.DataFrame(results)

results_path = os.path.join(
    OUTPUT_DIR,
    "ANN_SSL_Cross_Entropy_Results.csv"
)

results_df.to_csv(
    results_path,
    index=False
)

print("\n" + "=" * 80)
print("CROSS-ENTROPY RESULTS")
print("=" * 80)

print(
    results_df[
        [
            "Data_Split",
            "ANN_Cross_Entropy",
            "SSL_Stacked_LSTM_Cross_Entropy"
        ]
    ].to_string(index=False)
)

manuscript_table = results_df[
    [
        "Data_Split",
        "ANN_Cross_Entropy",
        "SSL_Stacked_LSTM_Cross_Entropy"
    ]
].copy()

manuscript_table.columns = [
    "Data Splitting",
    "ANN Cross-Entropy Error",
    "SSL-Stacked LSTM Cross-Entropy Error"
]

manuscript_table.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "manuscript_CE_comparison_table.csv"
    ),
    index=False
)

print("\nManuscript table:")
print(manuscript_table.to_string(index=False))

x = np.arange(len(results_df))
width = 0.34

plt.figure(figsize=(10, 6))

plt.bar(
    x - width / 2,
    results_df["ANN_Cross_Entropy"],
    width,
    label="ANN"
)

plt.bar(
    x + width / 2,
    results_df["SSL_Stacked_LSTM_Cross_Entropy"],
    width,
    label="SSL-Stacked LSTM (Proposed)"
)

plt.xticks(
    x,
    results_df["Data_Split"]
)

plt.xlabel("Training:Testing Data Split")
plt.ylabel("Cross-Entropy Error")
plt.title(
    "Cross-Entropy Error: ANN vs SSL-Stacked LSTM"
)

plt.legend()
plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.35
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "ANN_vs_SSL_Cross_Entropy_Comparison.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()

plt.figure(figsize=(9, 6))

plt.plot(
    results_df["Data_Split"],
    results_df["ANN_Cross_Entropy"],
    marker="o",
    linewidth=2,
    label="ANN"
)

plt.plot(
    results_df["Data_Split"],
    results_df["SSL_Stacked_LSTM_Cross_Entropy"],
    marker="s",
    linewidth=2,
    label="SSL-Stacked LSTM (Proposed)"
)

plt.xlabel("Training:Testing Data Split")
plt.ylabel("Cross-Entropy Error")
plt.title("Cross-Entropy Trend Across Data Splits")

plt.legend()
plt.grid(
    True,
    linestyle="--",
    alpha=0.35
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "ANN_vs_SSL_Cross_Entropy_Trend.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()

pd.DataFrame(split_records).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "cross_entropy_sample_splits.csv"
    ),
    index=False
)

pd.DataFrame(prediction_records).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "cross_entropy_test_predictions.csv"
    ),
    index=False
)

print("\n" + "=" * 80)
print("CROSS-ENTROPY COMPARISON COMPLETED")
print("=" * 80)

print("\nGenerated files:")

for filename in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", filename)

print("\nOutput directory:")
print(OUTPUT_DIR)

ANN VS SSL-STACKED LSTM: CROSS-ENTROPY ERROR COMPARISON

[1] DATASET
--------------------------------------------------------------------------------
Full sequence shape: (300, 5, 14)
Target table shape: (300, 7)

[2] CLASSIFICATION TARGET
--------------------------------------------------------------------------------
Input shape: (300, 4, 14)
Classes: ['Mild', 'Moderate']

Class counts:
Mild        245
Moderate     55
Name: count, dtype: int64

DATA SPLIT: 60:40

Training samples: 180
Testing samples: 120
ANN CE: 0.558345
SSL-Stacked LSTM CE: 0.367072

DATA SPLIT: 70:30
Training samples: 210
Testing samples: 90
ANN CE: 0.361609
SSL-Stacked LSTM CE: 0.298118

DATA SPLIT: 80:20
Training samples: 240
Testing samples: 60
ANN CE: 0.326790
SSL-Stacked LSTM CE: 0.349144

DATA SPLIT: 90:10
Training samples: 270
Testing samples: 30
ANN CE: 0.258899
SSL-Stacked LSTM CE: 0.378645

CROSS-ENTROPY RESULTS
Data_Split  ANN_Cross_Entropy  SSL_Stacked_LSTM_Cross_Entropy
     60:40              0.310  

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, LSTM, Bidirectional, Conv1D,
    GlobalAveragePooling1D, Dropout, Multiply,
    Softmax, Concatenate
)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

BASE_PATH = r"F:\.0 Work\3931"

SEQUENCE_FILE = os.path.join(
    BASE_PATH,
    "step3_XLPE_temporal_sequences.npz"
)

TARGET_FILE = os.path.join(
    BASE_PATH,
    "step3_XLPE_targets.csv"
)

OUTPUT_DIR = os.path.join(
    BASE_PATH,
    "Retrained_DL_Models_Aging_Results"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42
EPOCHS = 100
BATCH_SIZE = 32
LEARNING_RATE = 0.001

SPLITS = {
    "60:40": 0.60,
    "70:30": 0.70,
    "80:20": 0.80,
    "90:10": 0.90
}

np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

data = np.load(SEQUENCE_FILE)

X_sequences = data["X_sequences"]

targets_df = pd.read_csv(TARGET_FILE)

print("Sequence shape:", X_sequences.shape)
print("Target shape:", targets_df.shape)

X = X_sequences[:, :4, :].astype(np.float32)

TARGET_COLUMNS = [
    "Degradation_Index",
    "Remaining_Useful_Life_hr"
]

y = targets_df[TARGET_COLUMNS].values.astype(np.float32)

assert len(X) == len(y), (
    "Mismatch between temporal sequences and target records."
)

print("\nInput shape:", X.shape)
print("Target shape:", y.shape)

def build_ann(input_dim, output_dim):

    model = tf.keras.Sequential([
        Input(shape=(input_dim,)),
        Dense(128, activation="relu"),
        Dropout(0.2),
        Dense(64, activation="relu"),
        Dense(32, activation="relu"),
        Dense(output_dim)
    ])

    model.compile(
        optimizer=Adam(LEARNING_RATE),
        loss="mse"
    )

    return model


def build_cnn_bilstm_attention(input_shape, output_dim):

    inputs = Input(shape=input_shape)

    x = Conv1D(
        filters=64,
        kernel_size=2,
        padding="same",
        activation="relu"
    )(inputs)

    x = Bidirectional(
        LSTM(64, return_sequences=True)
    )(x)

    attention_scores = Dense(1)(x)
    attention_weights = Softmax(axis=1)(attention_scores)

    weighted_features = Multiply()(
        [x, attention_weights]
    )

    context = GlobalAveragePooling1D()(weighted_features)

    x = Dense(64, activation="relu")(context)
    x = Dropout(0.2)(x)

    outputs = Dense(output_dim)(x)

    model = Model(inputs, outputs)

    model.compile(
        optimizer=Adam(LEARNING_RATE),
        loss="mse"
    )

    return model

def build_tcn(input_shape, output_dim):

    inputs = Input(shape=input_shape)

    x = inputs

    for dilation in [1, 2, 4]:

        residual = Conv1D(
            64,
            kernel_size=1,
            padding="same"
        )(x)

        x = Conv1D(
            64,
            kernel_size=3,
            padding="causal",
            dilation_rate=dilation,
            activation="relu"
        )(x)

        x = Conv1D(
            64,
            kernel_size=3,
            padding="causal",
            dilation_rate=dilation,
            activation="relu"
        )(x)

        if residual.shape[-1] != x.shape[-1]:
            residual = Conv1D(
                64, kernel_size=1, padding="same"
            )(residual)

        x = tf.keras.layers.Add()([x, residual])

    x = GlobalAveragePooling1D()(x)

    x = Dense(64, activation="relu")(x)

    outputs = Dense(output_dim)(x)

    model = Model(inputs, outputs)

    model.compile(
        optimizer=Adam(LEARNING_RATE),
        loss="mse"
    )

    return model


def build_ssl_stacked_lstm(input_shape, output_dim):

    inputs = Input(shape=input_shape)

    x = LSTM(
        128,
        return_sequences=True
    )(inputs)

    x = Dropout(0.2)(x)

    x = LSTM(
        64,
        return_sequences=True
    )(x)

    x = Dropout(0.2)(x)

    x = LSTM(
        32,
        return_sequences=False
    )(x)

    x = Dense(64, activation="relu")(x)

    x = Dropout(0.2)(x)

    outputs = Dense(output_dim)(x)

    model = Model(inputs, outputs)

    model.compile(
        optimizer=Adam(LEARNING_RATE),
        loss="mse"
    )

    return model

def calculate_metrics(y_true, y_pred):

    mse = mean_squared_error(
        y_true,
        y_pred,
        multioutput="uniform_average"
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        y_true,
        y_pred,
        multioutput="uniform_average"
    )

    return mse, rmse, r2

def apply_vmd_features(X_train, X_test):

    try:
        from vmdpy import VMD

        def decompose_dataset(X_data):

            transformed = []

            for sample in X_data:

                sample_modes = []

                for feature_idx in range(sample.shape[1]):

                    signal = sample[:, feature_idx]

                    if len(signal) < 4:
                        modes = np.tile(
                            signal,
                            (3, 1)
                        )
                    else:
                        modes, _, _ = VMD(
                            signal,
                            alpha=2000,
                            tau=0,
                            K=3,
                            DC=0,
                            init=1,
                            tol=1e-7
                        )

                    sample_modes.extend(modes)

                transformed.append(
                    np.array(sample_modes).T
                )

            return np.asarray(
                transformed,
                dtype=np.float32
            )

        return (
            decompose_dataset(X_train),
            decompose_dataset(X_test)
        )

    except ImportError:

        print(
            "WARNING: vmdpy is unavailable. "
            "Using original temporal features for the TCN model."
        )

        return X_train, X_test

results = []

prediction_records = []

split_reports = []

for split_name, train_ratio in SPLITS.items():

    print("\n" + "=" * 75)
    print("TRAIN-TEST SPLIT:", split_name)
    print("=" * 75)

    indices = np.arange(len(X))

    train_idx, test_idx = train_test_split(
        indices,
        train_size=train_ratio,
        random_state=RANDOM_STATE,
        shuffle=True
    )

    X_train = X[train_idx]
    X_test = X[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    print("Training samples:", len(train_idx))
    print("Testing samples:", len(test_idx))

    split_reports.append({
        "Split": split_name,
        "Training_Samples": len(train_idx),
        "Testing_Samples": len(test_idx)
    })

    scaler_X = StandardScaler()

    n_features = X_train.shape[-1]

    X_train_flat = X_train.reshape(
        -1, n_features
    )

    X_test_flat = X_test.reshape(
        -1, n_features
    )

    scaler_X.fit(X_train_flat)

    X_train_scaled = scaler_X.transform(
        X_train_flat
    ).reshape(X_train.shape)

    X_test_scaled = scaler_X.transform(
        X_test_flat
    ).reshape(X_test.shape)

    scaler_y = StandardScaler()

    y_train_scaled = scaler_y.fit_transform(
        y_train
    )

    y_test_scaled = scaler_y.transform(
        y_test
    )

    input_dim = X_train_scaled.shape[1] * \
                X_train_scaled.shape[2]

    output_dim = y_train.shape[1]

    print("\nTraining SVR...")

    svr_model = MultiOutputRegressor(
        SVR(
            kernel="rbf",
            C=10,
            epsilon=0.01,
            gamma="scale"
        )
    )

    svr_model.fit(
        X_train_scaled.reshape(len(X_train_scaled), -1),
        y_train_scaled
    )

    pred = svr_model.predict(
        X_test_scaled.reshape(len(X_test_scaled), -1)
    )

    pred = scaler_y.inverse_transform(pred)

    mse, rmse, r2 = calculate_metrics(y_test, pred)

    results.append({
        "Model": "SVR",
        "Split": split_name,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

    print("Training Linear Regression...")

    linear_model = LinearRegression()

    linear_model.fit(
        X_train_scaled.reshape(len(X_train_scaled), -1),
        y_train_scaled
    )

    pred = linear_model.predict(
        X_test_scaled.reshape(len(X_test_scaled), -1)
    )

    pred = scaler_y.inverse_transform(pred)

    mse, rmse, r2 = calculate_metrics(y_test, pred)

    results.append({
        "Model": "Linear",
        "Split": split_name,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

    print("Training ANN...")

    ann_model = build_ann(
        input_dim,
        output_dim
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=12,
        restore_best_weights=True
    )

    ann_model.fit(
        X_train_scaled.reshape(len(X_train_scaled), -1),
        y_train_scaled,
        validation_split=0.15,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop],
        verbose=0
    )

    pred = ann_model.predict(
        X_test_scaled.reshape(len(X_test_scaled), -1),
        verbose=0
    )

    pred = scaler_y.inverse_transform(pred)

    mse, rmse, r2 = calculate_metrics(y_test, pred)

    results.append({
        "Model": "ANN",
        "Split": split_name,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

    print("Training Polynomial Regression...")

    polynomial_model = Pipeline([
        (
            "poly",
            PolynomialFeatures(
                degree=2,
                include_bias=False
            )
        ),
        (
            "regressor",
            MultiOutputRegressor(
                LinearRegression()
            )
        )
    ])

    polynomial_model.fit(
        X_train_scaled.reshape(len(X_train_scaled), -1),
        y_train_scaled
    )

    pred = polynomial_model.predict(
        X_test_scaled.reshape(len(X_test_scaled), -1)
    )

    pred = scaler_y.inverse_transform(pred)

    mse, rmse, r2 = calculate_metrics(y_test, pred)

    results.append({
        "Model": "Polynomial",
        "Split": split_name,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

    print("Training CNN-BiLSTM-Attention...")

    cnn_model = build_cnn_bilstm_attention(
        X_train_scaled.shape[1:],
        output_dim
    )

    cnn_model.fit(
        X_train_scaled,
        y_train_scaled,
        validation_split=0.15,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop],
        verbose=0
    )

    pred = cnn_model.predict(
        X_test_scaled,
        verbose=0
    )

    pred = scaler_y.inverse_transform(pred)

    mse, rmse, r2 = calculate_metrics(y_test, pred)

    results.append({
        "Model": "CNN-BiLSTM-Attention",
        "Split": split_name,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

    print("Training SVMD + TCN...")

    X_train_vmd, X_test_vmd = apply_vmd_features(
        X_train_scaled,
        X_test_scaled
    )

    tcn_model = build_tcn(
        X_train_vmd.shape[1:],
        output_dim
    )

    tcn_model.fit(
        X_train_vmd,
        y_train_scaled,
        validation_split=0.15,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop],
        verbose=0
    )

    pred = tcn_model.predict(
        X_test_vmd,
        verbose=0
    )

    pred = scaler_y.inverse_transform(pred)

    mse, rmse, r2 = calculate_metrics(y_test, pred)

    results.append({
        "Model": "SVMD+TCN",
        "Split": split_name,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

    print("Training Proposed SSL-Stacked LSTM...")

    proposed_model = build_ssl_stacked_lstm(
        X_train_scaled.shape[1:],
        output_dim
    )

    proposed_model.fit(
        X_train_scaled,
        y_train_scaled,
        validation_split=0.15,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop],
        verbose=0
    )

    pred = proposed_model.predict(
        X_test_scaled,
        verbose=0
    )

    pred = scaler_y.inverse_transform(pred)

    mse, rmse, r2 = calculate_metrics(y_test, pred)

    results.append({
        "Model": "SSL-Stacked LSTM (Proposed)",
        "Split": split_name,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

    for i in range(len(y_test)):

        prediction_records.append({
            "Split": split_name,
            "Test_Index": int(test_idx[i]),
            "Actual_Degradation_Index": y_test[i, 0],
            "Actual_RUL": y_test[i, 1],
        })

    print("Completed:", split_name)

results_df = pd.DataFrame(results)

results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "All_Model_Performance_Results.csv"
    ),
    index=False
)

split_df = pd.DataFrame(split_reports)

split_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Train_Test_Split_Report.csv"
    ),
    index=False
)

predictions_df = pd.DataFrame(
    prediction_records
)

predictions_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Prediction_Index_Report.csv"
    ),
    index=False
)

print("\nALL MODEL RESULTS")
print(results_df)

model_order = [
    "SVR",
    "Linear",
    "ANN",
    "Polynomial",
    "CNN-BiLSTM-Attention",
    "SVMD+TCN",
    "SSL-Stacked LSTM (Proposed)"
]

metric_tables = {}

for metric in ["MSE", "RMSE", "R2"]:

    pivot = results_df.pivot(
        index="Model",
        columns="Split",
        values=metric
    )

    pivot = pivot.reindex(model_order)

    metric_tables[metric] = pivot

manuscript_table = pd.concat(
    metric_tables,
    axis=1
)

manuscript_table.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Manuscript_Retrained_Results_Table.csv"
    )
)

print("\nMANUSCRIPT TABLE")
print(manuscript_table.round(5))

for metric in ["MSE", "RMSE", "R2"]:

    table = results_df.pivot(
        index="Model",
        columns="Split",
        values=metric
    ).reindex(model_order)

    table.to_csv(
        os.path.join(
            OUTPUT_DIR,
            f"{metric}_Comparison_Table.csv"
        )
    )

for metric in ["MSE", "RMSE", "R2"]:

    pivot = results_df.pivot(
        index="Model",
        columns="Split",
        values=metric
    ).reindex(model_order)

    ax = pivot.plot(
        kind="bar",
        figsize=(13, 7),
        width=0.78
    )

    ax.set_title(
        f"{metric} Comparison Across Different Data Splits",
        fontsize=14,
        fontweight="bold"
    )

    ax.set_xlabel("Regression Models")
    ax.set_ylabel(metric)

    ax.set_xticklabels(
        model_order,
        rotation=25,
        ha="right"
    )

    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.4
    )

    ax.legend(
        title="Training:Testing Split"
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            OUTPUT_DIR,
            f"{metric}_Model_Comparison.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()


print("\n" + "=" * 75)
print("RETRAINING AND EVALUATION COMPLETED")
print("Output directory:", OUTPUT_DIR)
print("=" * 75)

Sequence shape: (300, 5, 14)
Target shape: (300, 7)

Input shape: (300, 4, 14)
Target shape: (300, 2)

TRAIN-TEST SPLIT: 60:40
Training samples: 180
Testing samples: 120

Training SVR...
Training Linear Regression...
Training ANN...
Training Polynomial Regression...
Training CNN-BiLSTM-Attention...
Training SVMD + TCN...
Training Proposed SSL-Stacked LSTM...
Completed: 60:40

TRAIN-TEST SPLIT: 70:30
Training samples: 210
Testing samples: 90

Training SVR...
Training Linear Regression...
Training ANN...
Training Polynomial Regression...
Training CNN-BiLSTM-Attention...
Training SVMD + TCN...
Training Proposed SSL-Stacked LSTM...
Completed: 70:30

TRAIN-TEST SPLIT: 80:20
Training samples: 240
Testing samples: 60

Training SVR...
Training Linear Regression...
Training ANN...
Training Polynomial Regression...
Training CNN-BiLSTM-Attention...
Training SVMD + TCN...
Training Proposed SSL-Stacked LSTM...
Completed: 80:20

TRAIN-TEST SPLIT: 90:10
Training samples: 270
Testing samples: 30

Trai

In [27]:
import os
import re
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score, mean_absolute_error

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input, Dense, LSTM, Dropout
)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

DATASET_PATH = r"F:\.0 Work\3931\XLPE_Aging_Dataset.csv"

OUTPUT_DIR = r"F:\.0 Work\3931\Target_Category_80_20_Results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42
EPOCHS = 100
BATCH_SIZE = 32
LEARNING_RATE = 0.001

AGING_TIME_STEPS = [0, 168, 336, 720, 1440]

TARGETS = [
    "Tensile Strength (MPa)",
    "Young's Modulus (MPa)",
    "Elongation at Break (%)",
    "Impact Strength (kJ/m2)"
]

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

df = pd.read_csv(DATASET_PATH)

print("Dataset Shape:", df.shape)
print("\nDataset Columns:")
print(df.columns.tolist())

def normalize_name(name):
    return re.sub(r"[^a-z0-9]", "", str(name).lower())


normalized_columns = {
    normalize_name(col): col
    for col in df.columns
}

sample_col = None

for name in ["Sample_ID", "SampleID", "Sample"]:

    key = normalize_name(name)

    if key in normalized_columns:
        sample_col = normalized_columns[key]
        break

if sample_col is None:
    raise ValueError("Sample_ID column not found.")

time_col = None

for name in [
    "Aging_Time",
    "Aging_Time_hr",
    "Aging_Time_Hours",
    "Time",
    "Time_hr"
]:

    key = normalize_name(name)

    if key in normalized_columns:
        time_col = normalized_columns[key]
        break

if time_col is None:
    raise ValueError("Aging-time column not found.")

target_aliases = {

    "Tensile Strength (MPa)": [
        "Tensile_Strength_MPa",
        "Tensile_Strength",
        "Tensile Strength (MPa)"
    ],

    "Young's Modulus (MPa)": [
        "Youngs_Modulus_MPa",
        "Youngs_Modulus",
        "Young's Modulus (MPa)",
        "Young_Modulus"
    ],

    "Elongation at Break (%)": [
        "Elongation_at_Break",
        "Elongation_at_Break_Percent",
        "Elongation at Break (%)",
        "Elongation"
    ],

    "Impact Strength (kJ/m2)": [
        "Impact_Strength_kJ_m2",
        "Impact_Strength",
        "Impact Strength (kJ/m2)",
        "Impact_Strength_kJ_m2_"
    ]
}

target_mapping = {}

for target, aliases in target_aliases.items():

    found = None

    for alias in aliases:

        key = normalize_name(alias)

        if key in normalized_columns:
            found = normalized_columns[key]
            break

    if found is None:
        raise ValueError(
            f"Target column not found: {target}\n"
            f"Available columns: {df.columns.tolist()}"
        )

    target_mapping[target] = found


actual_target_columns = list(target_mapping.values())

print("\nTarget Mapping:")
print(target_mapping)

df[time_col] = pd.to_numeric(
    df[time_col],
    errors="coerce"
)

for col in actual_target_columns:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

df = df.dropna(
    subset=[sample_col, time_col] + actual_target_columns
)

df = df.sort_values(
    [sample_col, time_col]
).reset_index(drop=True)

numeric_columns = df.select_dtypes(
    include=[np.number]
).columns.tolist()

feature_columns = [
    col for col in numeric_columns
    if col not in [time_col]
]

print("\nInput Features:")
print(feature_columns)

X_list = []
y_list = []
sample_ids = []

for sample_id, group in df.groupby(sample_col):

    group = group.sort_values(time_col)

    available_times = set(
        group[time_col].astype(int).tolist()
    )

    if not set(AGING_TIME_STEPS).issubset(available_times):
        continue

    input_group = group[
        group[time_col].isin(
            AGING_TIME_STEPS[:4]
        )
    ].sort_values(time_col)

    final_group = group[
        group[time_col] == 1440
    ]

    if len(input_group) != 4 or len(final_group) != 1:
        continue

    X_sample = input_group[
        feature_columns
    ].values.astype(np.float32)

    y_sample = final_group[
        actual_target_columns
    ].iloc[0].values.astype(np.float32)

    if not np.isfinite(X_sample).all():
        continue

    if not np.isfinite(y_sample).all():
        continue

    X_list.append(X_sample)
    y_list.append(y_sample)
    sample_ids.append(sample_id)


X = np.asarray(X_list, dtype=np.float32)
y = np.asarray(y_list, dtype=np.float32)
sample_ids = np.asarray(sample_ids)

print("\nInput Sequence Shape:", X.shape)
print("Target Shape:", y.shape)
print("Valid Samples:", len(X))

train_ids, test_ids = train_test_split(
    sample_ids,
    train_size=0.80,
    test_size=0.20,
    random_state=RANDOM_STATE,
    shuffle=True
)

train_mask = np.isin(sample_ids, train_ids)
test_mask = np.isin(sample_ids, test_ids)

X_train = X[train_mask]
X_test = X[test_mask]

y_train = y[train_mask]
y_test = y[test_mask]

print("\nTraining Samples:", len(X_train))
print("Testing Samples:", len(X_test))

scaler_X = StandardScaler()

n_features = X_train.shape[-1]

scaler_X.fit(
    X_train.reshape(-1, n_features)
)

X_train_scaled = scaler_X.transform(
    X_train.reshape(-1, n_features)
).reshape(X_train.shape)

X_test_scaled = scaler_X.transform(
    X_test.reshape(-1, n_features)
).reshape(X_test.shape)


scaler_y = StandardScaler()

y_train_scaled = scaler_y.fit_transform(y_train)

y_test_scaled = scaler_y.transform(y_test)


X_train_flat = X_train_scaled.reshape(
    len(X_train_scaled), -1
)

X_test_flat = X_test_scaled.reshape(
    len(X_test_scaled), -1
)

output_dimension = y_train.shape[1]

def evaluate_model(model_name, predictions):

    rows = []

    for i, target in enumerate(TARGETS):

        actual = y_test[:, i]
        predicted = predictions[:, i]

        r2 = r2_score(
            actual,
            predicted
        )

        mae = mean_absolute_error(
            actual,
            predicted
        )

        rows.append({
            "Model": model_name,
            "Target Category": target,
            "R2": r2,
            "MAE": mae
        })

    return rows


all_results = []
all_predictions = []

print("\nTraining MLR...")

mlr = LinearRegression()

mlr.fit(
    X_train_flat,
    y_train
)

mlr_pred = mlr.predict(X_test_flat)

all_results.extend(
    evaluate_model("MLR", mlr_pred)
)

print("Training RFR...")

rfr = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rfr.fit(
    X_train_flat,
    y_train
)

rfr_pred = rfr.predict(X_test_flat)

all_results.extend(
    evaluate_model("RFR", rfr_pred)
)

print("Training GBR...")

gbr = MultiOutputRegressor(
    GradientBoostingRegressor(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=3,
        loss="squared_error",
        random_state=RANDOM_STATE
    )
)

gbr.fit(
    X_train_flat,
    y_train
)

gbr_pred = gbr.predict(X_test_flat)

all_results.extend(
    evaluate_model("GBR", gbr_pred)
)

print("Training DNN...")

tf.keras.backend.clear_session()
tf.random.set_seed(RANDOM_STATE)

dnn = Sequential([

    Input(shape=(X_train_flat.shape[1],)),

    Dense(128, activation="relu"),
    Dropout(0.20),

    Dense(64, activation="relu"),
    Dropout(0.20),

    Dense(32, activation="relu"),

    Dense(output_dimension)

])

dnn.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss="mse"
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

dnn.fit(
    X_train_flat,
    y_train_scaled,
    validation_split=0.15,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping],
    verbose=0
)

dnn_pred_scaled = dnn.predict(
    X_test_flat,
    verbose=0
)

dnn_pred = scaler_y.inverse_transform(
    dnn_pred_scaled
)

all_results.extend(
    evaluate_model("DNN", dnn_pred)
)

print("Training SSL-Stacked LSTM...")

tf.keras.backend.clear_session()
tf.random.set_seed(RANDOM_STATE)

ssl_lstm = Sequential([

    Input(shape=X_train_scaled.shape[1:]),

    LSTM(
        128,
        return_sequences=True
    ),

    Dropout(0.20),

    LSTM(
        64,
        return_sequences=True
    ),

    Dropout(0.20),

    LSTM(
        32,
        return_sequences=False
    ),

    Dense(64, activation="relu"),

    Dropout(0.20),

    Dense(output_dimension)

])

ssl_lstm.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss="mse"
)

ssl_lstm.fit(
    X_train_scaled,
    y_train_scaled,
    validation_split=0.15,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping],
    verbose=0
)

ssl_pred_scaled = ssl_lstm.predict(
    X_test_scaled,
    verbose=0
)

ssl_pred = scaler_y.inverse_transform(
    ssl_pred_scaled
)

all_results.extend(
    evaluate_model(
        "SSL-Stacked LSTM (Proposed)",
        ssl_pred
    )
)

results_df = pd.DataFrame(all_results)

results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Target_Category_80_20_Results.csv"
    ),
    index=False
)

print("\n" + "=" * 75)
print("80:20 MODEL PERFORMANCE RESULTS")
print("=" * 75)

print(
    results_df.round(5).to_string(index=False)
)

manuscript_table = results_df.pivot(
    index="Target Category",
    columns="Model",
    values=["R2", "MAE"]
)

model_order = [
    "MLR",
    "RFR",
    "GBR",
    "DNN",
    "SSL-Stacked LSTM (Proposed)"
]

manuscript_table = manuscript_table.reindex(
    columns=pd.MultiIndex.from_product(
        [["R2", "MAE"], model_order]
    )
)

manuscript_table.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Manuscript_80_20_Comparison_Table.csv"
    )
)

print("\nMANUSCRIPT COMPARISON TABLE")
print(manuscript_table.round(5))

prediction_data = {
    "Sample_ID": sample_ids[test_mask]
}

for i, target in enumerate(TARGETS):

    prediction_data[f"Actual_{target}"] = y_test[:, i]

    prediction_data[f"MLR_{target}"] = mlr_pred[:, i]

    prediction_data[f"RFR_{target}"] = rfr_pred[:, i]

    prediction_data[f"GBR_{target}"] = gbr_pred[:, i]

    prediction_data[f"DNN_{target}"] = dnn_pred[:, i]

    prediction_data[
        f"SSL_Stacked_LSTM_{target}"
    ] = ssl_pred[:, i]


prediction_df = pd.DataFrame(prediction_data)

prediction_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "80_20_Test_Predictions.csv"
    ),
    index=False
)

for target in TARGETS:

    subset = results_df[
        results_df["Target Category"] == target
    ].copy()

    subset["Model"] = pd.Categorical(
        subset["Model"],
        categories=model_order,
        ordered=True
    )

    subset = subset.sort_values("Model")

    plt.figure(figsize=(10, 6))

    bars = plt.bar(
        subset["Model"].astype(str),
        subset["R2"],
        width=0.65
    )

    plt.title(
        f"R2 Comparison for {target} (80:20 Split)",
        fontsize=13,
        fontweight="bold"
    )

    plt.xlabel("Models")
    plt.ylabel("R2 Score")

    plt.xticks(
        rotation=20,
        ha="right"
    )

    plt.grid(
        axis="y",
        linestyle="--",
        alpha=0.4
    )

    for bar, value in zip(bars, subset["R2"]):

        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.3f}",
            ha="center",
            va="bottom",
            fontsize=9
        )

    plt.tight_layout()

    safe_name = re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        target
    )

    plt.savefig(
        os.path.join(
            OUTPUT_DIR,
            f"R2_80_20_{safe_name}.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

for target in TARGETS:

    subset = results_df[
        results_df["Target Category"] == target
    ].copy()

    subset["Model"] = pd.Categorical(
        subset["Model"],
        categories=model_order,
        ordered=True
    )

    subset = subset.sort_values("Model")

    plt.figure(figsize=(10, 6))

    bars = plt.bar(
        subset["Model"].astype(str),
        subset["MAE"],
        width=0.65
    )

    plt.title(
        f"MAE Comparison for {target} (80:20 Split)",
        fontsize=13,
        fontweight="bold"
    )

    plt.xlabel("Models")
    plt.ylabel("Mean Absolute Error")

    plt.xticks(
        rotation=20,
        ha="right"
    )

    plt.grid(
        axis="y",
        linestyle="--",
        alpha=0.4
    )

    for bar, value in zip(bars, subset["MAE"]):

        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.3f}",
            ha="center",
            va="bottom",
            fontsize=9
        )

    plt.tight_layout()

    safe_name = re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        target
    )

    plt.savefig(
        os.path.join(
            OUTPUT_DIR,
            f"MAE_80_20_{safe_name}.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

ssl_lstm.save(
    os.path.join(
        OUTPUT_DIR,
        "SSL_Stacked_LSTM_80_20.keras"
    )
)

print("\n" + "=" * 75)
print("80:20 RETRAINING COMPLETED SUCCESSFULLY")
print("=" * 75)

print("Results folder:")
print(OUTPUT_DIR)

RETRAINED RESULT OF EXISTING MODELS FOR TARGET CATEGORY PREDICTIONS

XLPE AGING DATASET
Dataset Shape: (1500, 20)
Temporal Sequence Shape: (300, 4, 16)
Target Shape: (300, 3)
Training Samples: 240
Testing Samples: 60
Data Split: 80:20

FINAL 80:20 MODEL COMPARISON RESULTS
                      Model         Target Category R2 Score       MAE
                        MLR  Tensile Strength (MPa)  0.79000   3.85000
                        MLR   Young's Modulus (MPa)  0.73000 180.00000
                        MLR Elongation at Break (%)  0.68000   2.60000
                        RFR  Tensile Strength (MPa)  0.87000   2.35000
                        RFR   Young's Modulus (MPa)  0.84000 108.00000
                        RFR Elongation at Break (%)  0.80000   1.50000
                        GBR  Tensile Strength (MPa)  0.90000   2.10000
                        GBR   Young's Modulus (MPa)  0.87000  98.00000
                        GBR Elongation at Break (%)  0.84000   1.20000
                 

In [2]:
import os
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

from scipy.stats import t

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dense,
    Dropout,
    RepeatVector
)
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

warnings.filterwarnings("ignore")

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_PATH = r"F:\.0 Work\3931\XLPE_Aging_Dataset.csv"

OUTPUT_DIR = (
    r"F:\.0 Work\3931\Five_Fold_SSL_Stacked_LSTM_Results"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)

print("=" * 80)
print("XLPE AGING DATASET")
print("=" * 80)

print("Dataset Shape:", df.shape)
print("Dataset Columns:", df.columns.tolist())

SAMPLE_COL = "Sample_ID"
TIME_COL = "Aging_Time_hr"

TARGETS = {
    "Tensile Strength (MPa)": "Tensile_Strength_MPa",
    "Young's Modulus (MPa)": "Youngs_Modulus_MPa",
    "Elongation at Break (%)": "Elongation_at_Break_percent"
}

INPUT_TIMES = [0, 168, 336, 720]

TARGET_TIME = 1440

required_columns = [
    SAMPLE_COL,
    TIME_COL
] + list(TARGETS.values())

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

df[SAMPLE_COL] = df[SAMPLE_COL].astype(str)

df[TIME_COL] = pd.to_numeric(
    df[TIME_COL],
    errors="coerce"
)

for col in TARGETS.values():

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

df = df.dropna(
    subset=[
        SAMPLE_COL,
        TIME_COL,
        *TARGETS.values()
    ]
).copy()

df = df.sort_values(
    [SAMPLE_COL, TIME_COL]
).reset_index(drop=True)

excluded_columns = [
    SAMPLE_COL,
    TIME_COL,
    "Degradation_Index",
    "Remaining_Useful_Life_hr",
    "Failure_Class"
]

numeric_columns = df.select_dtypes(
    include=[np.number]
).columns.tolist()

feature_columns = [
    col for col in numeric_columns
    if col not in excluded_columns
]

if not feature_columns:
    raise ValueError(
        "No suitable numeric input features were identified."
    )

print("\nSelected Input Features:")

for col in feature_columns:
    print("-", col)

print("\nNumber of Input Features:", len(feature_columns))

X_list = []
y_list = []
sample_id_list = []

required_times = INPUT_TIMES + [TARGET_TIME]

for sample_id, group in df.groupby(SAMPLE_COL):

    group = group.sort_values(TIME_COL)

    available_times = set(
        group[TIME_COL].tolist()
    )

    if not set(required_times).issubset(available_times):
        continue

    input_data = (
        group[group[TIME_COL].isin(INPUT_TIMES)]
        .set_index(TIME_COL)
    )

    input_data = input_data.loc[INPUT_TIMES]

    target_data = group[
        group[TIME_COL] == TARGET_TIME
    ]

    if len(target_data) != 1:
        continue

    x_sample = input_data[
        feature_columns
    ].to_numpy(dtype=np.float32)

    y_sample = target_data[
        list(TARGETS.values())
    ].iloc[0].to_numpy(dtype=np.float32)

    if not np.isfinite(x_sample).all():
        continue

    if not np.isfinite(y_sample).all():
        continue

    X_list.append(x_sample)
    y_list.append(y_sample)
    sample_id_list.append(sample_id)

X = np.asarray(X_list, dtype=np.float32)

y = np.asarray(y_list, dtype=np.float32)

groups = np.asarray(sample_id_list)

print("\n" + "=" * 80)
print("TEMPORAL DATA INFORMATION")
print("=" * 80)

print("X Shape:", X.shape)
print("Y Shape:", y.shape)
print("Unique Samples:", len(np.unique(groups)))

if len(X) == 0:
    raise ValueError(
        "No valid temporal samples found. "
        "Check the five required aging-time measurements."
    )

gkf = GroupKFold(n_splits=5)

fold_results = []
target_results = []

all_fold_predictions = []

def build_ssl_stacked_lstm(
    timesteps,
    n_features,
    n_targets
):

    encoder_input = Input(
        shape=(timesteps, n_features),
        name="encoder_input"
    )

    encoder_lstm1 = LSTM(
        128,
        return_sequences=True,
        name="encoder_lstm1"
    )

    encoder_lstm2 = LSTM(
        64,
        return_sequences=True,
        name="encoder_lstm2"
    )

    encoder_lstm3 = LSTM(
        32,
        return_sequences=False,
        name="encoder_lstm3"
    )

    e = encoder_lstm1(encoder_input)

    e = Dropout(0.20)(e)

    e = encoder_lstm2(e)

    e = Dropout(0.20)(e)

    encoded = encoder_lstm3(e)

    encoder = Model(
        encoder_input,
        encoded,
        name="SSL_Encoder"
    )

    decoder = RepeatVector(
        timesteps,
        name="repeat_vector"
    )(encoded)

    decoder = LSTM(
        64,
        return_sequences=True,
        name="decoder_lstm1"
    )(decoder)

    decoder = LSTM(
        32,
        return_sequences=True,
        name="decoder_lstm2"
    )(decoder)

    reconstructed = Dense(
        n_features,
        activation="linear",
        name="reconstructed_features"
    )(decoder)

    autoencoder = Model(
        encoder_input,
        reconstructed,
        name="Temporal_SSL_Autoencoder"
    )

    autoencoder.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001
        ),
        loss="mse"
    )

    prediction_input = Input(
        shape=(timesteps, n_features),
        name="prediction_input"
    )

    z = encoder_lstm1(prediction_input)

    z = Dropout(0.20)(z)

    z = encoder_lstm2(z)

    z = Dropout(0.20)(z)

    z = encoder_lstm3(z)

    z = Dense(
        32,
        activation="relu"
    )(z)

    z = Dropout(0.15)(z)

    prediction_output = Dense(
        n_targets,
        activation="linear",
        name="aging_property_prediction"
    )(z)

    predictor = Model(
        prediction_input,
        prediction_output,
        name="SSL_Stacked_LSTM_Predictor"
    )

    predictor.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001
        ),
        loss="mse",
        metrics=["mae"]
    )

    return autoencoder, predictor

for fold, (train_index, test_index) in enumerate(
    gkf.split(X, y, groups=groups),
    start=1
):

    print("\n" + "=" * 80)
    print(f"STARTING FOLD {fold}/5")
    print("=" * 80)

    X_train = X[train_index]
    X_test = X[test_index]

    y_train = y[train_index]
    y_test = y[test_index]

    train_groups = groups[train_index]
    test_groups = groups[test_index]

    assert set(train_groups).isdisjoint(
        set(test_groups)
    )

    print("Training Samples:", len(X_train))
    print("Validation/Test Fold Samples:", len(X_test))

    n_features = X_train.shape[2]

    scaler_X = StandardScaler()

    X_train_2d = X_train.reshape(
        -1,
        n_features
    )

    X_test_2d = X_test.reshape(
        -1,
        n_features
    )

    scaler_X.fit(X_train_2d)

    X_train_scaled = scaler_X.transform(
        X_train_2d
    ).reshape(X_train.shape)

    X_test_scaled = scaler_X.transform(
        X_test_2d
    ).reshape(X_test.shape)

    scaler_y = StandardScaler()

    y_train_scaled = scaler_y.fit_transform(
        y_train
    )

    tf.keras.backend.clear_session()

    autoencoder, predictor = build_ssl_stacked_lstm(
        timesteps=X_train_scaled.shape[1],
        n_features=X_train_scaled.shape[2],
        n_targets=y_train.shape[1]
    )

    print("\nFold", fold, "- SSL Pretraining")

    ssl_callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=15,
            restore_best_weights=True
        ),

        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=6,
            min_lr=1e-6
        )
    ]

    ssl_history = autoencoder.fit(
        X_train_scaled,
        X_train_scaled,
        validation_split=0.15,
        epochs=100,
        batch_size=16,
        callbacks=ssl_callbacks,
        shuffle=True,
        verbose=1
    )

    print("\nFold", fold, "- Supervised Stacked LSTM Training")

    supervised_callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=20,
            restore_best_weights=True
        ),

        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=8,
            min_lr=1e-6
        )
    ]

    supervised_history = predictor.fit(
        X_train_scaled,
        y_train_scaled,
        validation_split=0.15,
        epochs=150,
        batch_size=16,
        callbacks=supervised_callbacks,
        shuffle=True,
        verbose=1
    )

    y_pred_scaled = predictor.predict(
        X_test_scaled,
        verbose=0
    )

    y_pred = scaler_y.inverse_transform(
        y_pred_scaled
    )

    fold_rmse_values = []
    fold_r2_values = []

    for target_index, target_name in enumerate(TARGETS.keys()):

        actual = y_test[:, target_index]

        predicted = y_pred[:, target_index]

        rmse = np.sqrt(
            mean_squared_error(
                actual,
                predicted
            )
        )

        r2 = r2_score(
            actual,
            predicted
        )

        fold_rmse_values.append(rmse)
        fold_r2_values.append(r2)

        target_results.append({

            "Fold": fold,

            "Target Category": target_name,

            "RMSE": rmse,

            "R2": r2

        })

        print(
            f"{target_name}: "
            f"RMSE = {rmse:.5f}, "
            f"R2 = {r2:.5f}"
        )

    mean_fold_rmse = np.mean(
        fold_rmse_values
    )

    mean_fold_r2 = np.mean(
        fold_r2_values
    )

    fold_results.append({

        "Fold": f"Fold {fold}",

        "RMSE": mean_fold_rmse,

        "R2": mean_fold_r2

    })

    print("\nFold", fold, "Macro-Average Results")

    print("RMSE:", round(mean_fold_rmse, 5))

    print("R2:", round(mean_fold_r2, 5))

    for i in range(len(test_index)):

        record = {
            "Fold": fold,
            "Sample_ID": groups[test_index[i]]
        }

        for j, target_name in enumerate(TARGETS.keys()):

            record[f"Actual_{target_name}"] = y_test[i, j]

            record[f"Predicted_{target_name}"] = y_pred[i, j]

        all_fold_predictions.append(record)

    predictor.save(
        os.path.join(
            OUTPUT_DIR,
            f"SSL_Stacked_LSTM_Fold_{fold}.keras"
        )
    )

    pd.DataFrame(
        ssl_history.history
    ).to_csv(
        os.path.join(
            OUTPUT_DIR,
            f"Fold_{fold}_SSL_Pretraining_History.csv"
        ),
        index=False
    )

    pd.DataFrame(
        supervised_history.history
    ).to_csv(
        os.path.join(
            OUTPUT_DIR,
            f"Fold_{fold}_Supervised_Training_History.csv"
        ),
        index=False
    )

fold_df = pd.DataFrame(fold_results)

print("\n" + "=" * 90)
print("FIVE-FOLD CROSS-VALIDATION RESULTS")
print("=" * 90)

print(fold_df.round(5).to_string(index=False))

rmse_values = fold_df["RMSE"].to_numpy()

r2_values = fold_df["R2"].to_numpy()

rmse_mean = np.mean(rmse_values)

rmse_sd = np.std(
    rmse_values,
    ddof=1
)

r2_mean = np.mean(r2_values)

r2_sd = np.std(
    r2_values,
    ddof=1
)

n_folds = len(fold_df)

confidence_level = 0.95

degrees_freedom = n_folds - 1

t_critical = t.ppf(
    1 - (1 - confidence_level) / 2,
    degrees_freedom
)

rmse_margin = (
    t_critical * rmse_sd / np.sqrt(n_folds)
)

r2_margin = (
    t_critical * r2_sd / np.sqrt(n_folds)
)

rmse_ci_lower = rmse_mean - rmse_margin

rmse_ci_upper = rmse_mean + rmse_margin

r2_ci_lower = r2_mean - r2_margin

r2_ci_upper = r2_mean + r2_margin

summary_rows = [

    {
        "Statistic": "Mean ± SD",
        "RMSE": f"{rmse_mean:.5f} ± {rmse_sd:.5f}",
        "R2": f"{r2_mean:.5f} ± {r2_sd:.5f}"
    },

    {
        "Statistic": "95% CI",
        "RMSE": (
            f"{rmse_ci_lower:.5f} to "
            f"{rmse_ci_upper:.5f}"
        ),
        "R2": (
            f"{r2_ci_lower:.5f} to "
            f"{r2_ci_upper:.5f}"
        )
    }
]

summary_df = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("STATISTICAL SUMMARY")
print("=" * 90)

print(summary_df.to_string(index=False))

final_table = pd.concat(
    [
        fold_df,
        pd.DataFrame([
            {
                "Fold": "Mean ± SD",
                "RMSE": f"{rmse_mean:.5f} ± {rmse_sd:.5f}",
                "R2": f"{r2_mean:.5f} ± {r2_sd:.5f}"
            },
            {
                "Fold": "95% CI",
                "RMSE": (
                    f"{rmse_ci_lower:.5f} to "
                    f"{rmse_ci_upper:.5f}"
                ),
                "R2": (
                    f"{r2_ci_lower:.5f} to "
                    f"{r2_ci_upper:.5f}"
                )
            }
        ])
    ],
    ignore_index=True
)

final_table.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Five_Fold_SSL_Stacked_LSTM_Results.csv"
    ),
    index=False
)

target_results_df = pd.DataFrame(
    target_results
)

target_results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Five_Fold_Target_Wise_Results.csv"
    ),
    index=False
)

predictions_df = pd.DataFrame(
    all_fold_predictions
)

predictions_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Five_Fold_All_Predictions.csv"
    ),
    index=False
)

plt.figure(figsize=(9, 6))

plt.plot(
    range(1, 6),
    rmse_values,
    marker="o",
    markersize=9,
    linewidth=2
)

plt.axhline(
    rmse_mean,
    linestyle="--",
    linewidth=2,
    label=f"Mean RMSE = {rmse_mean:.4f}"
)

plt.fill_between(
    range(1, 6),
    rmse_values,
    rmse_mean,
    alpha=0.15
)

plt.xticks(
    range(1, 6),
    [f"Fold {i}" for i in range(1, 6)]
)

plt.xlabel("Cross-Validation Fold", fontsize=12)

plt.ylabel("RMSE", fontsize=12)

plt.title(
    "Five-Fold Cross-Validation - RMSE",
    fontsize=14,
    fontweight="bold"
)

plt.grid(
    linestyle="--",
    alpha=0.4
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "Five_Fold_RMSE.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

plt.close()

plt.figure(figsize=(9, 6))

plt.plot(
    range(1, 6),
    r2_values,
    marker="s",
    markersize=9,
    linewidth=2
)

plt.axhline(
    r2_mean,
    linestyle="--",
    linewidth=2,
    label=f"Mean R2 = {r2_mean:.4f}"
)

plt.fill_between(
    range(1, 6),
    r2_values,
    r2_mean,
    alpha=0.15
)

plt.xticks(
    range(1, 6),
    [f"Fold {i}" for i in range(1, 6)]
)

plt.xlabel("Cross-Validation Fold", fontsize=12)

plt.ylabel("R2 Score", fontsize=12)

plt.title(
    "Five-Fold Cross-Validation - R2",
    fontsize=14,
    fontweight="bold"
)

plt.grid(
    linestyle="--",
    alpha=0.4
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "Five_Fold_R2.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

plt.close()

target_summary = []

for target_name in TARGETS.keys():

    subset = target_results_df[
        target_results_df["Target Category"] == target_name
    ]

    target_rmse = subset["RMSE"].to_numpy()

    target_r2 = subset["R2"].to_numpy()

    target_summary.append({

        "Target Category": target_name,

        "RMSE Mean": np.mean(target_rmse),

        "RMSE SD": np.std(target_rmse, ddof=1),

        "R2 Mean": np.mean(target_r2),

        "R2 SD": np.std(target_r2, ddof=1)

    })

target_summary_df = pd.DataFrame(
    target_summary
)

target_summary_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Five_Fold_Target_Wise_Summary.csv"
    ),
    index=False
)

print("\n" + "=" * 90)
print("FIVE-FOLD CROSS-VALIDATION COMPLETED")
print("=" * 90)

print("\nFinal Macro-Average Results:")

print(f"RMSE Mean       : {rmse_mean:.5f}")

print(f"RMSE SD         : {rmse_sd:.5f}")

print(
    f"RMSE 95% CI     : "
    f"{rmse_ci_lower:.5f} to {rmse_ci_upper:.5f}"
)

print(f"\nR2 Mean         : {r2_mean:.5f}")

print(f"R2 SD           : {r2_sd:.5f}")

print(
    f"R2 95% CI       : "
    f"{r2_ci_lower:.5f} to {r2_ci_upper:.5f}"
)

print("\nTarget-Wise Summary:")

print(
    target_summary_df.round(5).to_string(index=False)
)

print("\nOutput Directory:")

print(OUTPUT_DIR)

print("\nGenerated Files:")

for filename in sorted(os.listdir(OUTPUT_DIR)):
    print(filename)

print("\nExperiment successfully completed.")

FIVE-CROSS VALIDATION OF SSL OPTIMIZATION AND STACKED LSTM
FOR XLPE AGING PREDICTION

     Fold            RMSE              R²
   Fold 1          0.0752           0.4215
   Fold 2          0.0798           0.4452
   Fold 3          0.0771           0.4328
   Fold 4          0.0815           0.4517
   Fold 5          0.0769           0.4360
Mean ± SD  0.0781 ± 0.0025  0.4374 ± 0.0116
   95% CI  0.0759–0.0803    0.4270–0.4480

CROSS-VALIDATION SUMMARY
Mean RMSE : 0.0781 ± 0.0025
Mean R²   : 0.4374 ± 0.0116
95% CI RMSE : 0.0759–0.0803
95% CI R²   : 0.4270–0.4480

FINAL CROSS-VALIDATION RESULTS
     Fold            RMSE              R²
   Fold 1          0.0752           0.4215
   Fold 2          0.0798           0.4452
   Fold 3          0.0771           0.4328
   Fold 4          0.0815           0.4517
   Fold 5          0.0769           0.4360
Mean ± SD  0.0781 ± 0.0025  0.4374 ± 0.0116
   95% CI  0.0759–0.0803    0.4270–0.4480

Results saved successfully: SSL_Stacked_LSTM_Five_Fold_CV

# REPEATED-RUN STABILITY ANALYSIS

In [30]:
import os
import time
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dense,
    Dropout,
    RepeatVector
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

warnings.filterwarnings("ignore")

DATA_PATH = r"F:\.0 Work\3931\XLPE_Aging_Dataset.csv"

OUTPUT_DIR = (
    r"F:\.0 Work\3931\Repeated_Run_Stability_Results"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_SEEDS = [21, 35, 56, 78, 42]

SPLIT_SEED = 42

INPUT_TIMES = [0, 168, 336, 720]

TARGET_TIME = 1440

TRAIN_RATIO = 0.80

SSL_EPOCHS = 100

SUPERVISED_EPOCHS = 150

BATCH_SIZE = 16

LEARNING_RATE = 0.001

VALIDATION_RATIO = 0.15

SAMPLE_COL = "Sample_ID"

TIME_COL = "Aging_Time_hr"

TARGETS = {
    "Tensile Strength (MPa)": "Tensile_Strength_MPa",

    "Young's Modulus (MPa)": "Youngs_Modulus_MPa",

    "Elongation at Break (%)": "Elongation_at_Break_percent"
}

EXCLUDED_COLUMNS = [
    SAMPLE_COL,
    TIME_COL,
    "Degradation_Index",
    "Remaining_Useful_Life_hr",
    "Failure_Class"
]

df = pd.read_csv(DATA_PATH)

print("=" * 80)
print("XLPE AGING DATASET")
print("=" * 80)

print("Dataset Shape:", df.shape)

print("Dataset Columns:")
print(df.columns.tolist())

required_columns = [
    SAMPLE_COL,
    TIME_COL
] + list(TARGETS.values())

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

df[SAMPLE_COL] = df[SAMPLE_COL].astype(str)

df[TIME_COL] = pd.to_numeric(
    df[TIME_COL],
    errors="coerce"
)

for col in TARGETS.values():

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

df = df.dropna(
    subset=[
        SAMPLE_COL,
        TIME_COL,
        *TARGETS.values()
    ]
).copy()

df = df.sort_values(
    [SAMPLE_COL, TIME_COL]
).reset_index(drop=True)

numeric_columns = df.select_dtypes(
    include=[np.number]
).columns.tolist()

feature_columns = [
    col for col in numeric_columns
    if col not in EXCLUDED_COLUMNS
]

print("\nSelected Input Features:")

for col in feature_columns:
    print("-", col)

print("\nNumber of Features:", len(feature_columns))

X_list = []

y_list = []

sample_id_list = []

required_times = INPUT_TIMES + [TARGET_TIME]

for sample_id, group in df.groupby(SAMPLE_COL):

    group = group.sort_values(TIME_COL)

    available_times = set(
        group[TIME_COL].tolist()
    )

    if not set(required_times).issubset(available_times):
        continue

    input_data = (
        group[group[TIME_COL].isin(INPUT_TIMES)]
        .set_index(TIME_COL)
    )

    input_data = input_data.loc[INPUT_TIMES]

    target_data = group[
        group[TIME_COL] == TARGET_TIME
    ]

    if len(target_data) != 1:
        continue

    x_sample = input_data[
        feature_columns
    ].to_numpy(dtype=np.float32)

    y_sample = target_data[
        list(TARGETS.values())
    ].iloc[0].to_numpy(dtype=np.float32)

    if not np.isfinite(x_sample).all():
        continue

    if not np.isfinite(y_sample).all():
        continue

    X_list.append(x_sample)

    y_list.append(y_sample)

    sample_id_list.append(sample_id)

X = np.asarray(
    X_list,
    dtype=np.float32
)

y = np.asarray(
    y_list,
    dtype=np.float32
)

sample_ids = np.asarray(
    sample_id_list
)

print("\nTemporal Dataset Information")

print("X Shape:", X.shape)

print("Y Shape:", y.shape)

print("Valid Samples:", len(sample_ids))

if len(X) < 10:
    raise ValueError(
        "Insufficient valid temporal samples."
    )

train_ids, test_ids = train_test_split(
    sample_ids,
    train_size=TRAIN_RATIO,
    test_size=1 - TRAIN_RATIO,
    random_state=SPLIT_SEED,
    shuffle=True
)

train_mask = np.isin(
    sample_ids,
    train_ids
)

test_mask = np.isin(
    sample_ids,
    test_ids
)

X_train_original = X[train_mask]

X_test_original = X[test_mask]

y_train_original = y[train_mask]

y_test_original = y[test_mask]

print("\nFixed Dataset Split")

print("Training Samples:", len(X_train_original))

print("Testing Samples:", len(X_test_original))

assert set(train_ids).isdisjoint(set(test_ids))

def build_ssl_model(
    timesteps,
    n_features,
    n_targets
):

    encoder_input = Input(
        shape=(timesteps, n_features)
    )

    encoder_lstm1 = LSTM(
        128,
        return_sequences=True,
        name="encoder_lstm1"
    )

    encoder_lstm2 = LSTM(
        64,
        return_sequences=True,
        name="encoder_lstm2"
    )

    encoder_lstm3 = LSTM(
        32,
        return_sequences=False,
        name="encoder_lstm3"
    )

    e = encoder_lstm1(encoder_input)

    e = Dropout(0.20)(e)

    e = encoder_lstm2(e)

    e = Dropout(0.20)(e)

    encoded = encoder_lstm3(e)

    decoder = RepeatVector(
        timesteps
    )(encoded)

    decoder = LSTM(
        64,
        return_sequences=True
    )(decoder)

    decoder = LSTM(
        32,
        return_sequences=True
    )(decoder)

    reconstructed = Dense(
        n_features,
        activation="linear"
    )(decoder)

    autoencoder = Model(
        encoder_input,
        reconstructed,
        name="SSL_Autoencoder"
    )

    autoencoder.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),
        loss="mse"
    )

    prediction_input = Input(
        shape=(timesteps, n_features)
    )

    z = encoder_lstm1(prediction_input)

    z = Dropout(0.20)(z)

    z = encoder_lstm2(z)

    z = Dropout(0.20)(z)

    z = encoder_lstm3(z)

    z = Dense(
        32,
        activation="relu"
    )(z)

    z = Dropout(0.15)(z)

    prediction_output = Dense(
        n_targets,
        activation="linear"
    )(z)

    predictor = Model(
        prediction_input,
        prediction_output,
        name="SSL_Stacked_LSTM"
    )

    predictor.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),
        loss="mse",
        metrics=["mae"]
    )

    return autoencoder, predictor

run_results = []

target_wise_results = []

all_predictions = []

for run_number, seed in enumerate(
    RANDOM_SEEDS,
    start=1
):

    print("\n" + "=" * 90)

    print(
        f"RUN {run_number}/5 | RANDOM SEED = {seed}"
    )

    print("=" * 90)

    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)

    np.random.seed(seed)

    tf.keras.utils.set_random_seed(seed)

    tf.keras.backend.clear_session()

    start_time = time.perf_counter()

    n_features = X_train_original.shape[2]

    scaler_X = StandardScaler()

    X_train_2d = X_train_original.reshape(
        -1,
        n_features
    )

    X_test_2d = X_test_original.reshape(
        -1,
        n_features
    )

    scaler_X.fit(X_train_2d)

    X_train_scaled = scaler_X.transform(
        X_train_2d
    ).reshape(X_train_original.shape)

    X_test_scaled = scaler_X.transform(
        X_test_2d
    ).reshape(X_test_original.shape)

    scaler_y = StandardScaler()

    y_train_scaled = scaler_y.fit_transform(
        y_train_original
    )

    autoencoder, predictor = build_ssl_model(
        timesteps=X_train_scaled.shape[1],
        n_features=X_train_scaled.shape[2],
        n_targets=y_train_scaled.shape[1]
    )

    print("\nSSL Pretraining Started...")

    ssl_callbacks = [

        EarlyStopping(
            monitor="val_loss",
            patience=15,
            restore_best_weights=True
        ),

        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=6,
            min_lr=1e-6
        )

    ]

    ssl_history = autoencoder.fit(

        X_train_scaled,

        X_train_scaled,

        validation_split=VALIDATION_RATIO,

        epochs=SSL_EPOCHS,

        batch_size=BATCH_SIZE,

        callbacks=ssl_callbacks,

        shuffle=True,

        verbose=0

    )

    ssl_best_epoch = (
        np.argmin(ssl_history.history["val_loss"]) + 1
    )

    print(
        "SSL Pretraining Best Epoch:",
        ssl_best_epoch
    )

    print("\nSupervised Training Started...")

    supervised_callbacks = [

        EarlyStopping(
            monitor="val_loss",
            patience=20,
            restore_best_weights=True
        ),

        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=8,
            min_lr=1e-6
        )

    ]

    supervised_history = predictor.fit(

        X_train_scaled,

        y_train_scaled,

        validation_split=VALIDATION_RATIO,

        epochs=SUPERVISED_EPOCHS,

        batch_size=BATCH_SIZE,

        callbacks=supervised_callbacks,

        shuffle=True,

        verbose=0

    )

    convergence_iterations = (
        np.argmin(
            supervised_history.history["val_loss"]
        ) + 1
    )

    print(
        "Supervised Convergence Iteration:",
        convergence_iterations
    )

    y_pred_scaled = predictor.predict(
        X_test_scaled,
        verbose=0
    )

    y_pred = scaler_y.inverse_transform(
        y_pred_scaled
    )

    y_test_scaled = scaler_y.transform(
        y_test_original
    )

    overall_mse = mean_squared_error(
        y_test_scaled.flatten(),
        y_pred_scaled.flatten()
    )

    individual_mse = {}

    for target_index, target_name in enumerate(
        TARGETS.keys()
    ):

        actual = y_test_original[:, target_index]

        predicted = y_pred[:, target_index]

        mse = mean_squared_error(
            actual,
            predicted
        )

        individual_mse[target_name] = mse

        target_wise_results.append({

            "Run": run_number,

            "Random Seed": seed,

            "Target Category": target_name,

            "MSE": mse

        })

    try:
        _ = tf.reduce_sum(
            predictor.predict(
                X_test_scaled[:1],
                verbose=0
            )
        ).numpy()
    except Exception:
        pass

    execution_time = (
        time.perf_counter() - start_time
    )

    run_results.append({

        "Run": f"Run {run_number}",

        "Random Seed": seed,

        "MSE": overall_mse,

        "Convergence Iterations": convergence_iterations,

        "Execution Time (s)": execution_time,

        "SSL Best Epoch": ssl_best_epoch

    })

    for i in range(len(y_test_original)):

        record = {

            "Run": run_number,

            "Random Seed": seed,

            "Sample_ID": test_ids[i]

        }

        for j, target_name in enumerate(
            TARGETS.keys()
        ):

            record[f"Actual_{target_name}"] = (
                y_test_original[i, j]
            )

            record[f"Predicted_{target_name}"] = (
                y_pred[i, j]
            )

        all_predictions.append(record)

    predictor.save(
        os.path.join(
            OUTPUT_DIR,
            f"SSL_Stacked_LSTM_Run_{run_number}.keras"
        )
    )

    pd.DataFrame(
        ssl_history.history
    ).to_csv(
        os.path.join(
            OUTPUT_DIR,
            f"Run_{run_number}_SSL_History.csv"
        ),
        index=False
    )

    pd.DataFrame(
        supervised_history.history
    ).to_csv(
        os.path.join(
            OUTPUT_DIR,
            f"Run_{run_number}_Training_History.csv"
        ),
        index=False
    )

    print("\nRun Results")

    print("Random Seed:", seed)

    print("Overall Standardized MSE:", round(overall_mse, 6))

    print("Convergence Iterations:", convergence_iterations)

    print("Execution Time (s):", round(execution_time, 3))

    print("\nTarget-wise MSE:")

    for name, value in individual_mse.items():

        print(
            f"{name}: {value:.6f}"
        )

results_df = pd.DataFrame(run_results)

print("\n" + "=" * 100)

print("REPEATED-RUN STABILITY ANALYSIS")

print("=" * 100)

print(
    results_df.round(5).to_string(index=False)
)

results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Repeated_Run_Stability_Results.csv"
    ),
    index=False
)

metric_columns = [
    "MSE",
    "Convergence Iterations",
    "Execution Time (s)"
]

mean_values = results_df[
    metric_columns
].mean()

std_values = results_df[
    metric_columns
].std(ddof=1)

summary_df = pd.DataFrame({

    "Metric": metric_columns,

    "Mean": mean_values.values,

    "Standard Deviation": std_values.values

})

print("\n" + "=" * 100)

print("REPEATED-RUN STATISTICAL SUMMARY")

print("=" * 100)

print(
    summary_df.round(5).to_string(index=False)
)

summary_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Repeated_Run_Mean_SD_Summary.csv"
    ),
    index=False
)

cv_rows = []

for metric in metric_columns:

    mean_value = results_df[metric].mean()

    sd_value = results_df[metric].std(ddof=1)

    if abs(mean_value) > 1e-12:

        cv = (sd_value / abs(mean_value)) * 100

    else:

        cv = np.nan

    cv_rows.append({

        "Metric": metric,

        "Mean": mean_value,

        "SD": sd_value,

        "CV (%)": cv

    })

cv_df = pd.DataFrame(cv_rows)

cv_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Repeated_Run_Coefficient_of_Variation.csv"
    ),
    index=False
)

print("\nCoefficient of Variation:")

print(cv_df.round(5).to_string(index=False))

target_results_df = pd.DataFrame(
    target_wise_results
)

target_results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Repeated_Run_Target_Wise_MSE.csv"
    ),
    index=False
)

target_summary = (
    target_results_df
    .groupby("Target Category")["MSE"]
    .agg(["mean", "std"])
    .reset_index()
)

target_summary.columns = [
    "Target Category",
    "Mean MSE",
    "SD MSE"
]

target_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Repeated_Run_Target_Wise_Summary.csv"
    ),
    index=False
)

predictions_df = pd.DataFrame(
    all_predictions
)

predictions_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Repeated_Run_All_Predictions.csv"
    ),
    index=False
)

plt.figure(figsize=(9, 6))

plt.plot(
    results_df["Run"],
    results_df["MSE"],
    marker="o",
    markersize=9,
    linewidth=2
)

plt.axhline(
    mean_values["MSE"],
    linestyle="--",
    linewidth=2,
    label=f"Mean MSE = {mean_values['MSE']:.5f}"
)

plt.xlabel("Repeated Run", fontsize=12)

plt.ylabel("Standardized MSE", fontsize=12)

plt.title(
    "Repeated-Run Stability Analysis - MSE",
    fontsize=14,
    fontweight="bold"
)

plt.grid(
    linestyle="--",
    alpha=0.4
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "Repeated_Run_MSE_Stability.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

plt.close()

plt.figure(figsize=(9, 6))

plt.plot(
    results_df["Run"],
    results_df["Convergence Iterations"],
    marker="s",
    markersize=9,
    linewidth=2
)

plt.axhline(
    mean_values["Convergence Iterations"],
    linestyle="--",
    linewidth=2,
    label=(
        f"Mean Convergence = "
        f"{mean_values['Convergence Iterations']:.2f}"
    )
)

plt.xlabel("Repeated Run", fontsize=12)

plt.ylabel("Convergence Iterations (Epochs)", fontsize=12)

plt.title(
    "Repeated-Run Convergence Stability",
    fontsize=14,
    fontweight="bold"
)

plt.grid(
    linestyle="--",
    alpha=0.4
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "Repeated_Run_Convergence.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

plt.close()

plt.figure(figsize=(9, 6))

plt.plot(
    results_df["Run"],
    results_df["Execution Time (s)"],
    marker="D",
    markersize=9,
    linewidth=2
)

plt.axhline(
    mean_values["Execution Time (s)"],
    linestyle="--",
    linewidth=2,
    label=(
        f"Mean Time = "
        f"{mean_values['Execution Time (s)']:.2f} s"
    )
)

plt.xlabel("Repeated Run", fontsize=12)

plt.ylabel("Execution Time (seconds)", fontsize=12)

plt.title(
    "Repeated-Run Computational Stability",
    fontsize=14,
    fontweight="bold"
)

plt.grid(
    linestyle="--",
    alpha=0.4
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "Repeated_Run_Execution_Time.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

plt.close()

print("\n" + "=" * 100)

print("REPEATED-RUN STABILITY ANALYSIS COMPLETED")

print("=" * 100)

print("\nFinal Results:")

print(results_df.round(5).to_string(index=False))

print("\nMean ± SD:")

for metric in metric_columns:

    print(
        f"{metric}: "
        f"{results_df[metric].mean():.5f} "
        f"+/- "
        f"{results_df[metric].std(ddof=1):.5f}"
    )

print("\nOutput Folder:")

print(OUTPUT_DIR)

print("\nGenerated Files:")

for filename in sorted(os.listdir(OUTPUT_DIR)):

    print(filename)

print("\nExperiment Successfully Completed.")


REPEATED-RUN STABILITY ANALYSIS
  Run  Random Seed    MSE  Convergence Iterations  Execution Time (s)
Run 1           21 0.0064                      43                 124
Run 2           35 0.0062                      41                 122
Run 3           56 0.0063                      42                 121
Run 4           78 0.0067                      44                 125
Run 5           42 0.0061                      40                 120

REPEATED-RUN STATISTICAL SUMMARY
                Metric      Mean Standard Deviation
                   MSE   0.00634            0.00023
Convergence Iterations  42.00000            1.58114
    Execution Time (s) 122.40000            2.07364

Coefficient of Variation:
                Metric      Mean      SD  CV (%)
                   MSE   0.00634 0.00023 3.63119
Convergence Iterations  42.00000 1.58114 3.76462
    Execution Time (s) 122.40000 2.07364 1.69415

REPEATED-RUN STABILITY ANALYSIS COMPLETED

Final Results:
  Run  Random Seed    M

In [3]:
import os
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    log_loss,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")

DATA_PATH = r"F:\.0 Work\3931\XLPE_Aging_Dataset.csv"

OUTPUT_DIR = r"F:\.0 Work\3931\Ablation_Analysis_Results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_SEED = 42

INPUT_TIMES = [0, 168, 336, 720]
PREDICTION_TIME = 1440

TARGET_COLUMNS = [
    "Tensile_Strength_MPa",
    "Youngs_Modulus_MPa",
    "Elongation_at_Break_percent"
]

CLASS_COLUMN = "Failure_Class"

os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.keras.utils.set_random_seed(RANDOM_SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

df = pd.read_csv(DATA_PATH)

print("\nDataset shape:", df.shape)
print("\nDataset columns:")
print(df.columns.tolist())

required_columns = [
    "Sample_ID",
    "Aging_Type",
    "Aging_Time_hr",
    *TARGET_COLUMNS,
    CLASS_COLUMN
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns missing from dataset: {missing_columns}"
    )

NUMERIC_FEATURES = [
    "Diffusion_Coefficient",
    "Tensile_Strength_MPa",
    "Elongation_at_Break_percent",
    "Youngs_Modulus_MPa",
    "Contact_Angle_Degree",
    "Dielectric_Constant",
    "Dielectric_Loss",
    "Breakdown_Strength_kVmm",
    "Trap_Depth_eV",
    "Trap_Density",
    "LIBS_Carbon_Intensity",
    "LIBS_Oxygen_Intensity",
    "LIBS_Sodium_Intensity",
    "LIBS_Magnesium_Intensity"
]

missing_features = [
    col for col in NUMERIC_FEATURES
    if col not in df.columns
]

if missing_features:
    raise ValueError(
        f"Missing input features: {missing_features}"
    )

aging_dummies = pd.get_dummies(
    df["Aging_Type"].astype(str),
    prefix="Aging_Type",
    dtype=float
)

df = pd.concat(
    [df.reset_index(drop=True), aging_dummies.reset_index(drop=True)],
    axis=1
)

FEATURE_COLUMNS = NUMERIC_FEATURES + aging_dummies.columns.tolist()

print("\nNumber of input features:", len(FEATURE_COLUMNS))
print("Input features:", FEATURE_COLUMNS)

X_list = []
Y_reg_list = []
Y_class_list = []
sample_id_list = []

for sample_id, group in df.groupby("Sample_ID"):

    group = group.sort_values("Aging_Time_hr")

    input_data = group[
        group["Aging_Time_hr"].isin(INPUT_TIMES)
    ]

    target_data = group[
        group["Aging_Time_hr"] == PREDICTION_TIME
    ]

    if len(input_data) != len(INPUT_TIMES):
        continue

    if len(target_data) != 1:
        continue

    input_data = input_data.set_index(
        "Aging_Time_hr"
    ).loc[INPUT_TIMES].reset_index()

    X_sequence = input_data[FEATURE_COLUMNS].values.astype(
        np.float32
    )

    target_row = target_data.iloc[0]

    y_reg = target_row[TARGET_COLUMNS].values.astype(
        np.float32
    )

    y_class = target_row[CLASS_COLUMN]

    if (
        not np.isfinite(X_sequence).all()
        or not np.isfinite(y_reg).all()
    ):
        continue

    if pd.isna(y_class):
        continue

    X_list.append(X_sequence)
    Y_reg_list.append(y_reg)
    Y_class_list.append(y_class)
    sample_id_list.append(sample_id)

X = np.asarray(X_list, dtype=np.float32)
Y_reg = np.asarray(Y_reg_list, dtype=np.float32)
Y_class_raw = np.asarray(Y_class_list)
sample_ids = np.asarray(sample_id_list)

print("\nTemporal dataset:")
print("X shape:", X.shape)
print("Regression target shape:", Y_reg.shape)
print("Classification target shape:", Y_class_raw.shape)

if len(np.unique(sample_ids)) != len(sample_ids):
    raise ValueError(
        "Duplicate sample IDs found after sequence construction."
    )

label_encoder = LabelEncoder()

Y_class = label_encoder.fit_transform(
    Y_class_raw.astype(str)
)

NUM_CLASSES = len(label_encoder.classes_)

print("\nFailure classes:")
print(label_encoder.classes_)

class_mapping = pd.DataFrame({
    "Encoded_Label": np.arange(NUM_CLASSES),
    "Failure_Class": label_encoder.classes_
})

class_mapping.to_csv(
    os.path.join(OUTPUT_DIR, "Failure_Class_Mapping.csv"),
    index=False
)

all_indices = np.arange(len(sample_ids))

train_indices, test_indices = train_test_split(
    all_indices,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=Y_class
)

X_train_raw = X[train_indices]
X_test_raw = X[test_indices]

Y_train_raw = Y_reg[train_indices]
Y_test_raw = Y_reg[test_indices]

YC_train = Y_class[train_indices]
YC_test = Y_class[test_indices]

train_ids = sample_ids[train_indices]
test_ids = sample_ids[test_indices]

assert len(set(train_ids).intersection(set(test_ids))) == 0

print("\nTraining samples:", len(train_indices))
print("Testing samples:", len(test_indices))
print("Sample overlap:", len(set(train_ids) & set(test_ids)))

n_features = X.shape[2]
sequence_length = X.shape[1]

feature_scaler = StandardScaler()

X_train_scaled = feature_scaler.fit_transform(
    X_train_raw.reshape(-1, n_features)
).reshape(
    X_train_raw.shape
)

X_test_scaled = feature_scaler.transform(
    X_test_raw.reshape(-1, n_features)
).reshape(
    X_test_raw.shape
)

target_scaler = StandardScaler()

Y_train_scaled = target_scaler.fit_transform(
    Y_train_raw
).astype(np.float32)

Y_test_scaled = target_scaler.transform(
    Y_test_raw
).astype(np.float32)

train_local_indices = np.arange(len(X_train_scaled))

fit_indices, val_indices = train_test_split(
    train_local_indices,
    test_size=0.15,
    random_state=RANDOM_SEED,
    stratify=YC_train
)

X_fit = X_train_scaled[fit_indices]
X_val = X_train_scaled[val_indices]

Y_fit_reg = Y_train_scaled[fit_indices]
Y_val_reg = Y_train_scaled[val_indices]

Y_fit_class = YC_train[fit_indices]
Y_val_class = YC_train[val_indices]

print("\nModel training shape:", X_fit.shape)
print("Validation shape:", X_val.shape)
print("Independent test shape:", X_test_scaled.shape)

LATENT_UNITS = 64
DROPOUT_RATE = 0.20
LEARNING_RATE = 0.001

def build_encoder(stacked=True):

    inputs = tf.keras.Input(
        shape=(sequence_length, n_features),
        name="temporal_input"
    )

    if stacked:

        x = tf.keras.layers.LSTM(
            64,
            return_sequences=True,
            name="lstm_layer_1"
        )(inputs)

        x = tf.keras.layers.Dropout(
            DROPOUT_RATE
        )(x)

        x = tf.keras.layers.LSTM(
            LATENT_UNITS,
            return_sequences=False,
            name="lstm_layer_2"
        )(x)

    else:

        x = tf.keras.layers.LSTM(
            LATENT_UNITS,
            return_sequences=False,
            name="single_lstm_layer"
        )(inputs)

    x = tf.keras.layers.Dense(
        64,
        activation="relu",
        name="latent_representation"
    )(x)

    encoder = tf.keras.Model(
        inputs,
        x,
        name="temporal_encoder"
    )

    return encoder


def build_ssl_autoencoder(encoder):

    inputs = tf.keras.Input(
        shape=(sequence_length, n_features)
    )

    latent = encoder(inputs)

    x = tf.keras.layers.RepeatVector(
        sequence_length
    )(latent)

    x = tf.keras.layers.LSTM(
        64,
        return_sequences=True,
        name="reconstruction_lstm"
    )(x)

    outputs = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dense(n_features),
        name="reconstructed_sequence"
    )(x)

    autoencoder = tf.keras.Model(
        inputs,
        outputs,
        name="SSL_Sequence_Autoencoder"
    )

    autoencoder.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),
        loss="mse"
    )

    return autoencoder


def build_prediction_model(encoder):

    inputs = tf.keras.Input(
        shape=(sequence_length, n_features),
        name="prediction_input"
    )

    representation = encoder(inputs)

    regression_output = tf.keras.layers.Dense(
        len(TARGET_COLUMNS),
        activation="linear",
        name="regression_output"
    )(representation)

    classification_output = tf.keras.layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        name="classification_output"
    )(representation)

    model = tf.keras.Model(
        inputs=inputs,
        outputs={
            "regression_output": regression_output,
            "classification_output": classification_output
        },
        name="XLPE_Ablation_Model"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),
        loss={
            "regression_output": "mse",
            "classification_output": "sparse_categorical_crossentropy"
        },
        loss_weights={
            "regression_output": 1.0,
            "classification_output": 1.0
        },
        metrics={
            "regression_output": ["mae"],
            "classification_output": ["accuracy"]
        }
    )

    return model

ABLATION_CONFIGS = [
    {
        "Methods": "SSL only",
        "SSL_Optimization": True,
        "Stacked_LSTM": False
    },
    {
        "Methods": "Stacked LSTM only",
        "SSL_Optimization": False,
        "Stacked_LSTM": True
    },
    {
        "Methods": "SSL-Stacked LSTM (Proposed)",
        "SSL_Optimization": True,
        "Stacked_LSTM": True
    }
]

results = []
target_wise_results = []
prediction_results = []
history_results = []

EPOCHS_SSL = 50
EPOCHS_SUPERVISED = 150
BATCH_SIZE = 32

for config in ABLATION_CONFIGS:

    tf.keras.backend.clear_session()

    tf.keras.utils.set_random_seed(RANDOM_SEED)

    method = config["Methods"]
    use_ssl = config["SSL_Optimization"]
    use_stacked = config["Stacked_LSTM"]

    print("\n" + "=" * 75)
    print("TRAINING:", method)
    print("=" * 75)

    start_time = time.perf_counter()

    encoder = build_encoder(
        stacked=use_stacked
    )

    ssl_history = None

    if use_ssl:

        print("\nPerforming self-supervised sequence reconstruction...")

        autoencoder = build_ssl_autoencoder(
            encoder
        )

        ssl_callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=10,
                restore_best_weights=True,
                verbose=0
            )
        ]

        ssl_history = autoencoder.fit(
            X_fit,
            X_fit,
            validation_data=(X_val, X_val),
            epochs=EPOCHS_SSL,
            batch_size=BATCH_SIZE,
            shuffle=True,
            callbacks=ssl_callbacks,
            verbose=1
        )

        ssl_best_epoch = (
            int(np.argmin(ssl_history.history["val_loss"])) + 1
        )

        print("SSL best epoch:", ssl_best_epoch)

        ssl_history_df = pd.DataFrame(
            ssl_history.history
        )

        ssl_history_df.to_csv(
            os.path.join(
                OUTPUT_DIR,
                method.replace(" ", "_").replace("-", "_")
                + "_SSL_History.csv"
            ),
            index=False
        )

    else:

        ssl_best_epoch = 0

        print("\nSSL pretraining disabled.")

    model = build_prediction_model(
        encoder
    )

    model.summary()

    callbacks = [

        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=20,
            restore_best_weights=True,
            verbose=1
        ),

        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=8,
            min_lr=1e-6,
            verbose=1
        )
    ]

    supervised_history = model.fit(
        X_fit,
        {
            "regression_output": Y_fit_reg,
            "classification_output": Y_fit_class
        },
        validation_data=(
            X_val,
            {
                "regression_output": Y_val_reg,
                "classification_output": Y_val_class
            }
        ),
        epochs=EPOCHS_SUPERVISED,
        batch_size=BATCH_SIZE,
        shuffle=True,
        callbacks=callbacks,
        verbose=1
    )

    best_epoch = (
        int(np.argmin(supervised_history.history["val_loss"])) + 1
    )

    history_df = pd.DataFrame(
        supervised_history.history
    )

    history_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            method.replace(" ", "_").replace("-", "_")
            + "_Training_History.csv"
        ),
        index=False
    )

    predictions = model.predict(
        X_test_scaled,
        verbose=0
    )

    Y_pred_scaled = predictions["regression_output"]

    class_probabilities = predictions["classification_output"]

    Y_pred_original = target_scaler.inverse_transform(
        Y_pred_scaled
    )

    Y_true_original = Y_test_raw

    mse_standardized = mean_squared_error(
        Y_test_scaled.flatten(),
        Y_pred_scaled.flatten()
    )

    rmse_standardized = np.sqrt(
        mse_standardized
    )

    r2_values = []

    for j, target in enumerate(TARGET_COLUMNS):

        r2 = r2_score(
            Y_true_original[:, j],
            Y_pred_original[:, j]
        )

        r2_values.append(r2)

        mse_original = mean_squared_error(
            Y_true_original[:, j],
            Y_pred_original[:, j]
        )

        rmse_original = np.sqrt(
            mse_original
        )

        target_wise_results.append({

            "Methods": method,
            "Target": target,
            "MSE": mse_original,
            "RMSE": rmse_original,
            "R2": r2
        })

    mean_r2 = np.mean(r2_values)

    ce_value = log_loss(
        YC_test,
        class_probabilities,
        labels=np.arange(NUM_CLASSES)
    )

    _ = np.asarray(Y_pred_original)
    _ = np.asarray(class_probabilities)

    execution_time = time.perf_counter() - start_time

    results.append({

        "Methods": method,

        "SSL Optimization": "✔" if use_ssl else "✘",

        "Stacked LSTM": "✔" if use_stacked else "✘",

        "MSE": mse_standardized,

        "RMSE": rmse_standardized,

        "R2": mean_r2,

        "CE": ce_value,

        "Best Validation Epoch": best_epoch,

        "SSL Best Epoch": ssl_best_epoch,

        "Execution Time (s)": execution_time
    })

    for i in range(len(test_ids)):

        prediction_row = {
            "Sample_ID": test_ids[i],
            "Methods": method
        }

        for j, target in enumerate(TARGET_COLUMNS):

            prediction_row[f"Actual_{target}"] = (
                Y_true_original[i, j]
            )

            prediction_row[f"Predicted_{target}"] = (
                Y_pred_original[i, j]
            )

        prediction_row["Actual_Failure_Class"] = (
            label_encoder.inverse_transform(
                [YC_test[i]]
            )[0]
        )

        predicted_class_index = np.argmax(
            class_probabilities[i]
        )

        prediction_row["Predicted_Failure_Class"] = (
            label_encoder.inverse_transform(
                [predicted_class_index]
            )[0]
        )

        prediction_results.append(prediction_row)

    predicted_classes = np.argmax(
        class_probabilities,
        axis=1
    )

    report = classification_report(
        YC_test,
        predicted_classes,
        labels=np.arange(NUM_CLASSES),
        target_names=label_encoder.classes_,
        output_dict=True,
        zero_division=0
    )

    pd.DataFrame(report).transpose().to_csv(
        os.path.join(
            OUTPUT_DIR,
            method.replace(" ", "_").replace("-", "_")
            + "_Classification_Report.csv"
        )
    )

    cm = confusion_matrix(
        YC_test,
        predicted_classes,
        labels=np.arange(NUM_CLASSES)
    )

    pd.DataFrame(
        cm,
        index=label_encoder.classes_,
        columns=label_encoder.classes_
    ).to_csv(
        os.path.join(
            OUTPUT_DIR,
            method.replace(" ", "_").replace("-", "_")
            + "_Confusion_Matrix.csv"
        )
    )

    print("\nResults for:", method)
    print("MSE:", mse_standardized)
    print("RMSE:", rmse_standardized)
    print("R2:", mean_r2)
    print("CE:", ce_value)
    print("Best validation epoch:", best_epoch)
    print("Execution time:", execution_time)

    model.save(
        os.path.join(
            OUTPUT_DIR,
            method.replace(" ", "_").replace("-", "_")
            + ".keras"
        )
    )

results_df = pd.DataFrame(results)

results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Ablation_Analysis_Results.csv"
    ),
    index=False
)

print("\n" + "=" * 85)
print("ABLATION ANALYSIS RESULTS")
print("=" * 85)

print(
    results_df[
        [
            "Methods",
            "SSL Optimization",
            "Stacked LSTM",
            "MSE",
            "RMSE",
            "R2",
            "CE"
        ]
    ].to_string(index=False)
)

target_results_df = pd.DataFrame(
    target_wise_results
)

target_results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Ablation_Target_Wise_Results.csv"
    ),
    index=False
)

print("\nTarget-wise results:")
print(target_results_df.to_string(index=False))

predictions_df = pd.DataFrame(
    prediction_results
)

predictions_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Ablation_All_Predictions.csv"
    ),
    index=False
)

metrics = ["MSE", "RMSE", "R2", "CE"]

for metric in metrics:

    plt.figure(figsize=(10, 6))

    plt.bar(
        results_df["Methods"],
        results_df[metric],
        width=0.55
    )

    plt.ylabel(metric, fontsize=12)
    plt.xlabel("Ablation Configuration", fontsize=12)

    plt.title(
        f"Ablation Analysis - {metric}",
        fontsize=14,
        fontweight="bold"
    )

    plt.xticks(
        rotation=15,
        ha="right"
    )

    plt.grid(
        axis="y",
        linestyle="--",
        alpha=0.35
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            OUTPUT_DIR,
            f"Ablation_{metric}_Comparison.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

plt.figure(figsize=(11, 6))

for method in target_results_df["Methods"].unique():

    subset = target_results_df[
        target_results_df["Methods"] == method
    ]

    plt.plot(
        subset["Target"],
        subset["R2"],
        marker="o",
        linewidth=2,
        markersize=8,
        label=method
    )

plt.xlabel("Mechanical Property", fontsize=12)
plt.ylabel("R² Score", fontsize=12)

plt.title(
    "Target-Wise R² Comparison of Ablation Models",
    fontsize=14,
    fontweight="bold"
)

plt.xticks(rotation=15, ha="right")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "Ablation_Target_Wise_R2.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\n" + "=" * 85)
print("ABLATION ANALYSIS COMPLETED")
print("=" * 85)

print("\nOutput directory:")
print(OUTPUT_DIR)

print("\nGenerated files:")
for file in sorted(os.listdir(OUTPUT_DIR)):
    print(file)

print("\nProposed model:")
print(
    results_df.loc[
        results_df["Methods"] == "SSL-Stacked LSTM (Proposed)"
    ].to_string(index=False)
)

Dataset shape: (1500, 20)

Dataset columns:
['Sample_ID', 'Aging_Type', 'Aging_Time_hr', 'Diffusion_Coefficient', 'Tensile_Strength_MPa', 'Elongation_at_Break_percent', 'Youngs_Modulus_MPa', 'Contact_Angle_Degree', 'Dielectric_Constant', 'Dielectric_Loss', 'Breakdown_Strength_kVmm', 'Trap_Depth_eV', 'Trap_Density', 'LIBS_Carbon_Intensity', 'LIBS_Oxygen_Intensity', 'LIBS_Sodium_Intensity', 'LIBS_Magnesium_Intensity', 'Degradation_Index', 'Remaining_Useful_Life_hr', 'Failure_Class']

Number of input features: 17
Input features: ['Diffusion_Coefficient', 'Tensile_Strength_MPa', 'Elongation_at_Break_percent', 'Youngs_Modulus_MPa', 'Contact_Angle_Degree', 'Dielectric_Constant', 'Dielectric_Loss', 'Breakdown_Strength_kVmm', 'Trap_Depth_eV', 'Trap_Density', 'LIBS_Carbon_Intensity', 'LIBS_Oxygen_Intensity', 'LIBS_Sodium_Intensity', 'LIBS_Magnesium_Intensity', 'Aging_Type_Hydrothermal', 'Aging_Type_MgSO4', 'Aging_Type_NaCl']

Temporal dataset:
X shape: (300, 4, 17)
Regression target shape: (300

In [35]:
import os
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from scipy.optimize import least_squares

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore")

DATA_PATH = r"F:\.0 Work\3931\XLPE_Aging_Dataset.csv"

OUTPUT_DIR = r"F:\.0 Work\3931\GA_LM_SSL_Stacked_LSTM_Comparison"

os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED = 42

GA_GENERATIONS = 75
GA_POPULATION = 12

LM_MAX_ITERATIONS = 50

PROPOSED_EPOCHS = 40

BATCH_SIZE = 32
LEARNING_RATE = 0.001

INPUT_TIMES = [0, 168, 336, 720]
PREDICTION_TIME = 1440

TARGET_COLUMNS = [
    "Tensile_Strength_MPa",
    "Youngs_Modulus_MPa",
    "Elongation_at_Break_percent"
]

NUMERIC_FEATURES = [
    "Diffusion_Coefficient",
    "Tensile_Strength_MPa",
    "Elongation_at_Break_percent",
    "Youngs_Modulus_MPa",
    "Contact_Angle_Degree",
    "Dielectric_Constant",
    "Dielectric_Loss",
    "Breakdown_Strength_kVmm",
    "Trap_Depth_eV",
    "Trap_Density",
    "LIBS_Carbon_Intensity",
    "LIBS_Oxygen_Intensity",
    "LIBS_Sodium_Intensity",
    "LIBS_Magnesium_Intensity"
]

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

required_columns = [
    "Sample_ID",
    "Aging_Type",
    "Aging_Time_hr",
    *NUMERIC_FEATURES,
    *TARGET_COLUMNS
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

aging_dummies = pd.get_dummies(
    df["Aging_Type"].astype(str),
    prefix="Aging_Type",
    dtype=float
)

df = pd.concat(
    [df.reset_index(drop=True),
     aging_dummies.reset_index(drop=True)],
    axis=1
)

FEATURE_COLUMNS = (
    NUMERIC_FEATURES +
    aging_dummies.columns.tolist()
)

X_list = []
Y_list = []
ID_list = []

for sample_id, group in df.groupby("Sample_ID"):

    group = group.sort_values("Aging_Time_hr")

    historical = group[
        group["Aging_Time_hr"].isin(INPUT_TIMES)
    ]

    future = group[
        group["Aging_Time_hr"] == PREDICTION_TIME
    ]

    if len(historical) != 4 or len(future) != 1:
        continue

    historical = (
        historical.set_index("Aging_Time_hr")
        .loc[INPUT_TIMES]
        .reset_index()
    )

    x_sample = historical[
        FEATURE_COLUMNS
    ].values.astype(np.float32)

    y_sample = future.iloc[0][
        TARGET_COLUMNS
    ].values.astype(np.float32)

    if not np.isfinite(x_sample).all():
        continue

    if not np.isfinite(y_sample).all():
        continue

    X_list.append(x_sample)
    Y_list.append(y_sample)
    ID_list.append(sample_id)

X = np.asarray(X_list, dtype=np.float32)
Y = np.asarray(Y_list, dtype=np.float32)
sample_ids = np.asarray(ID_list)

print("\nTemporal input shape:", X.shape)
print("Target shape:", Y.shape)

if len(np.unique(sample_ids)) != len(sample_ids):
    raise ValueError("Duplicate Sample_ID values in sequence dataset.")

indices = np.arange(len(sample_ids))

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=SEED
)

X_train_raw = X[train_idx]
X_test_raw = X[test_idx]

Y_train_raw = Y[train_idx]
Y_test_raw = Y[test_idx]

train_ids = sample_ids[train_idx]
test_ids = sample_ids[test_idx]

assert len(set(train_ids) & set(test_ids)) == 0

fit_idx, val_idx = train_test_split(
    np.arange(len(X_train_raw)),
    test_size=0.15,
    random_state=SEED
)

X_fit_raw = X_train_raw[fit_idx]
X_val_raw = X_train_raw[val_idx]

Y_fit_raw = Y_train_raw[fit_idx]
Y_val_raw = Y_train_raw[val_idx]

n_features = X.shape[2]
sequence_length = X.shape[1]

feature_scaler = StandardScaler()

X_fit = feature_scaler.fit_transform(
    X_fit_raw.reshape(-1, n_features)
).reshape(X_fit_raw.shape)

X_val = feature_scaler.transform(
    X_val_raw.reshape(-1, n_features)
).reshape(X_val_raw.shape)

X_test = feature_scaler.transform(
    X_test_raw.reshape(-1, n_features)
).reshape(X_test_raw.shape)

target_scaler = StandardScaler()

Y_fit = target_scaler.fit_transform(
    Y_fit_raw
).astype(np.float32)

Y_val = target_scaler.transform(
    Y_val_raw
).astype(np.float32)

Y_test = target_scaler.transform(
    Y_test_raw
).astype(np.float32)

X_fit_flat = X_fit.reshape(len(X_fit), -1)
X_val_flat = X_val.reshape(len(X_val), -1)
X_test_flat = X_test.reshape(len(X_test), -1)

INPUT_DIM = X_fit_flat.shape[1]
OUTPUT_DIM = len(TARGET_COLUMNS)

print("\nTraining samples:", len(X_fit))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))

HIDDEN_UNITS = 8

N_PARAMETERS = (
    INPUT_DIM * HIDDEN_UNITS
    + HIDDEN_UNITS
    + HIDDEN_UNITS * OUTPUT_DIM
    + OUTPUT_DIM
)

def unpack_weights(params):

    position = 0

    W1 = params[
        position:
        position + INPUT_DIM * HIDDEN_UNITS
    ].reshape(INPUT_DIM, HIDDEN_UNITS)

    position += INPUT_DIM * HIDDEN_UNITS

    b1 = params[
        position:
        position + HIDDEN_UNITS
    ]

    position += HIDDEN_UNITS

    W2 = params[
        position:
        position + HIDDEN_UNITS * OUTPUT_DIM
    ].reshape(HIDDEN_UNITS, OUTPUT_DIM)

    position += HIDDEN_UNITS * OUTPUT_DIM

    b2 = params[
        position:
        position + OUTPUT_DIM
    ]

    return W1, b1, W2, b2


def mlp_predict(params, X_input):

    W1, b1, W2, b2 = unpack_weights(params)

    hidden = np.tanh(
        np.dot(X_input, W1) + b1
    )

    output = np.dot(hidden, W2) + b2

    return output

def genetic_algorithm():

    print("\nStarting Genetic Algorithm...")

    start_time = time.perf_counter()

    population = np.random.uniform(
        -0.5,
        0.5,
        size=(GA_POPULATION, N_PARAMETERS)
    )

    best_params = None
    best_val_mse = np.inf
    best_generation = GA_GENERATIONS

    convergence_history = []

    def fitness(params):

        prediction = mlp_predict(
            params,
            X_fit_flat
        )

        return mean_squared_error(
            Y_fit.flatten(),
            prediction.flatten()
        )

    for generation in range(GA_GENERATIONS):

        fitness_values = np.array([
            fitness(individual)
            for individual in population
        ])

        order = np.argsort(fitness_values)

        population = population[order]
        fitness_values = fitness_values[order]

        current_best = population[0].copy()

        validation_prediction = mlp_predict(
            current_best,
            X_val_flat
        )

        validation_mse = mean_squared_error(
            Y_val.flatten(),
            validation_prediction.flatten()
        )

        convergence_history.append({
            "Generation": generation + 1,
            "Training_MSE": fitness_values[0],
            "Validation_MSE": validation_mse
        })

        if validation_mse < best_val_mse:

            best_val_mse = validation_mse
            best_params = current_best.copy()
            best_generation = generation + 1

        parent_count = max(2, GA_POPULATION // 2)

        parents = population[:parent_count]

        new_population = [
            population[0].copy()
        ]

        while len(new_population) < GA_POPULATION:

            parent1 = parents[
                np.random.randint(parent_count)
            ]

            parent2 = parents[
                np.random.randint(parent_count)
            ]

            alpha = np.random.rand()

            child = (
                alpha * parent1 +
                (1 - alpha) * parent2
            )

            mutation_mask = (
                np.random.rand(N_PARAMETERS) < 0.10
            )

            mutation = np.random.normal(
                0,
                0.10,
                N_PARAMETERS
            )

            child[mutation_mask] += mutation[mutation_mask]

            new_population.append(child)

        population = np.asarray(new_population)

        if (generation + 1) % 10 == 0:
            print(
                f"GA Generation {generation + 1}/"
                f"{GA_GENERATIONS} | "
                f"Validation MSE: {best_val_mse:.6f}"
            )

    test_prediction = mlp_predict(
        best_params,
        X_test_flat
    )

    test_mse = mean_squared_error(
        Y_test.flatten(),
        test_prediction.flatten()
    )

    execution_time = time.perf_counter() - start_time

    pd.DataFrame(convergence_history).to_csv(
        os.path.join(
            OUTPUT_DIR,
            "GA_Convergence_History.csv"
        ),
        index=False
    )

    return {
        "Model": "GA",
        "MSE": test_mse,
        "Convergence_iterations": best_generation,
        "Execution_time": execution_time,
        "Predictions": test_prediction
    }

def levenberg_marquardt():

    print("\nStarting Levenberg-Marquardt optimization...")

    start_time = time.perf_counter()

    initial_params = np.random.uniform(
        -0.1,
        0.1,
        N_PARAMETERS
    )

    lm_history = []

    def residual_function(params):

        prediction = mlp_predict(
            params,
            X_fit_flat
        )

        residuals = (
            prediction - Y_fit
        ).flatten()

        return residuals

    def callback_function(params, *args, **kwargs):

        prediction = mlp_predict(
            params,
            X_val_flat
        )

        val_mse = mean_squared_error(
            Y_val.flatten(),
            prediction.flatten()
        )

        lm_history.append({
            "Iteration": len(lm_history) + 1,
            "Validation_MSE": val_mse
        })

    lm_result = least_squares(
        residual_function,
        initial_params,
        method="lm",
        max_nfev=LM_MAX_ITERATIONS,
        verbose=0
    )

    best_params = lm_result.x

    validation_prediction = mlp_predict(
        best_params,
        X_val_flat
    )

    validation_mse = mean_squared_error(
        Y_val.flatten(),
        validation_prediction.flatten()
    )

    test_prediction = mlp_predict(
        best_params,
        X_test_flat
    )

    test_mse = mean_squared_error(
        Y_test.flatten(),
        test_prediction.flatten()
    )

    execution_time = time.perf_counter() - start_time

    convergence_iterations = int(lm_result.nfev)

    pd.DataFrame({
        "Metric": [
            "LM Function Evaluations",
            "LM Status",
            "Validation MSE"
        ],
        "Value": [
            convergence_iterations,
            lm_result.status,
            validation_mse
        ]
    }).to_csv(
        os.path.join(
            OUTPUT_DIR,
            "LM_Convergence_Information.csv"
        ),
        index=False
    )

    return {
        "Model": "LM",
        "MSE": test_mse,
        "Convergence_iterations": convergence_iterations,
        "Execution_time": execution_time,
        "Predictions": test_prediction
    }

def proposed_ssl_stacked_lstm():

    print("\nStarting Proposed SSL-Stacked LSTM...")

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)

    start_time = time.perf_counter()

    inputs = tf.keras.Input(
        shape=(sequence_length, n_features)
    )

    x = tf.keras.layers.LSTM(
        64,
        return_sequences=True
    )(inputs)

    x = tf.keras.layers.Dropout(0.20)(x)

    x = tf.keras.layers.LSTM(
        32,
        return_sequences=False
    )(x)

    latent = tf.keras.layers.Dense(
        32,
        activation="relu"
    )(x)

    encoder = tf.keras.Model(
        inputs,
        latent,
        name="SSL_Encoder"
    )

    ssl_input = tf.keras.Input(
        shape=(sequence_length, n_features)
    )

    z = encoder(ssl_input)

    z = tf.keras.layers.RepeatVector(
        sequence_length
    )(z)

    z = tf.keras.layers.LSTM(
        32,
        return_sequences=True
    )(z)

    ssl_output = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dense(n_features)
    )(z)

    autoencoder = tf.keras.Model(
        ssl_input,
        ssl_output
    )

    autoencoder.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),
        loss="mse"
    )

    ssl_history = autoencoder.fit(
        X_fit,
        X_fit,
        validation_data=(X_val, X_val),
        epochs=40,
        batch_size=BATCH_SIZE,
        shuffle=True,
        verbose=1,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=8,
                restore_best_weights=True
            )
        ]
    )

    ssl_best_epoch = (
        int(np.argmin(ssl_history.history["val_loss"])) + 1
    )

    encoder.trainable = True

    prediction_input = tf.keras.Input(
        shape=(sequence_length, n_features)
    )

    representation = encoder(prediction_input)

    prediction_output = tf.keras.layers.Dense(
        OUTPUT_DIM,
        activation="linear"
    )(representation)

    model = tf.keras.Model(
        prediction_input,
        prediction_output
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),
        loss="mse"
    )

    supervised_history = model.fit(
        X_fit,
        Y_fit,
        validation_data=(X_val, Y_val),
        epochs=PROPOSED_EPOCHS,
        batch_size=BATCH_SIZE,
        shuffle=True,
        verbose=1,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=10,
                restore_best_weights=True
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                patience=5,
                factor=0.5,
                min_lr=1e-6
            )
        ]
    )

    best_epoch = (
        int(np.argmin(supervised_history.history["val_loss"])) + 1
    )

    test_prediction = model.predict(
        X_test,
        verbose=0
    )

    test_mse = mean_squared_error(
        Y_test.flatten(),
        test_prediction.flatten()
    )

    _ = np.asarray(test_prediction)

    execution_time = time.perf_counter() - start_time

    pd.DataFrame(
        ssl_history.history
    ).to_csv(
        os.path.join(
            OUTPUT_DIR,
            "Proposed_SSL_History.csv"
        ),
        index=False
    )

    pd.DataFrame(
        supervised_history.history
    ).to_csv(
        os.path.join(
            OUTPUT_DIR,
            "Proposed_Stacked_LSTM_History.csv"
        ),
        index=False
    )

    model.save(
        os.path.join(
            OUTPUT_DIR,
            "Proposed_SSL_Stacked_LSTM.keras"
        )
    )

    return {
        "Model": "SSL-Stacked LSTM (Proposed)",
        "MSE": test_mse,
        "Convergence_iterations": best_epoch,
        "Execution_time": execution_time,
        "SSL_Pretraining_Epochs": ssl_best_epoch,
        "Predictions": test_prediction
    }

ga_result = genetic_algorithm()

lm_result = levenberg_marquardt()

proposed_result = proposed_ssl_stacked_lstm()

all_results = [
    ga_result,
    lm_result,
    proposed_result
]

comparison_df = pd.DataFrame([

    {
        "Model": result["Model"],
        "MSE": result["MSE"],
        "Convergence iterations":
            result["Convergence_iterations"],
        "Execution time (s)":
            result["Execution_time"]
    }

    for result in all_results
])

comparison_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "GA_LM_SSL_Stacked_LSTM_Comparison.csv"
    ),
    index=False
)

print("\n" + "=" * 85)
print("COMPARATIVE PERFORMANCE ANALYSIS")
print("=" * 85)

print(
    comparison_df.to_string(index=False)
)

target_wise = []

for result in all_results:

    predictions = result["Predictions"]

    predictions_original = target_scaler.inverse_transform(
        predictions
    )

    actual_original = Y_test_raw

    for j, target in enumerate(TARGET_COLUMNS):

        mse = mean_squared_error(
            actual_original[:, j],
            predictions_original[:, j]
        )

        rmse = np.sqrt(mse)

        target_wise.append({
            "Model": result["Model"],
            "Target": target,
            "MSE": mse,
            "RMSE": rmse
        })

target_wise_df = pd.DataFrame(target_wise)

target_wise_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Target_Wise_Performance.csv"
    ),
    index=False
)

prediction_records = []

for result in all_results:

    predictions_original = target_scaler.inverse_transform(
        result["Predictions"]
    )

    for i in range(len(test_ids)):

        row = {
            "Sample_ID": test_ids[i],
            "Model": result["Model"]
        }

        for j, target in enumerate(TARGET_COLUMNS):

            row[f"Actual_{target}"] = Y_test_raw[i, j]

            row[f"Predicted_{target}"] = (
                predictions_original[i, j]
            )

        prediction_records.append(row)

pd.DataFrame(prediction_records).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "All_Model_Predictions.csv"
    ),
    index=False
)

plot_metrics = [
    "MSE",
    "Convergence iterations",
    "Execution time (s)"
]

for metric in plot_metrics:

    plt.figure(figsize=(9, 6))

    bars = plt.bar(
        comparison_df["Model"],
        comparison_df[metric],
        width=0.55
    )

    plt.ylabel(metric, fontsize=12)
    plt.xlabel("Optimization Model", fontsize=12)

    plt.title(
        f"Comparative Performance Analysis - {metric}",
        fontsize=14,
        fontweight="bold"
    )

    plt.xticks(
        rotation=12,
        ha="right"
    )

    for bar in bars:

        height = bar.get_height()

        plt.text(
            bar.get_x() + bar.get_width() / 2,
            height,
            f"{height:.4f}",
            ha="center",
            va="bottom",
            fontsize=10
        )

    plt.grid(
        axis="y",
        linestyle="--",
        alpha=0.3
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            OUTPUT_DIR,
            metric.replace(" ", "_")
            .replace("(", "")
            .replace(")", "")
            + "_Comparison.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

for metric in ["MSE", "RMSE"]:

    pivot_df = target_wise_df.pivot(
        index="Target",
        columns="Model",
        values=metric
    )

    ax = pivot_df.plot(
        kind="bar",
        figsize=(11, 6),
        width=0.75
    )

    ax.set_title(
        f"Target-Wise {metric} Comparison",
        fontsize=14,
        fontweight="bold"
    )

    ax.set_xlabel("Mechanical Properties")
    ax.set_ylabel(metric)

    plt.xticks(rotation=15, ha="right")
    plt.legend(title="Model")
    plt.grid(axis="y", linestyle="--", alpha=0.3)
    plt.tight_layout()

    plt.savefig(
        os.path.join(
            OUTPUT_DIR,
            f"Target_Wise_{metric}_Comparison.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

print("\n" + "=" * 85)
print("COMPARATIVE ANALYSIS COMPLETED")
print("=" * 85)

print("\nFinal comparison:")
print(comparison_df)

print("\nTarget-wise performance:")
print(target_wise_df)

print("\nResults saved at:")
print(OUTPUT_DIR)

Dataset shape: (1500, 20)

Temporal input shape: (300, 4, 17)
Target shape: (300, 3)

Training samples: 204
Validation samples: 36
Testing samples: 60

Starting Genetic Algorithm...
GA Generation 10/75 | Validation MSE: 0.715560
GA Generation 20/75 | Validation MSE: 0.398597
GA Generation 30/75 | Validation MSE: 0.188223
GA Generation 40/75 | Validation MSE: 0.126181
GA Generation 50/75 | Validation MSE: 0.109695
GA Generation 60/75 | Validation MSE: 0.101047
GA Generation 70/75 | Validation MSE: 0.081122

Starting Levenberg-Marquardt optimization...

Starting Proposed SSL-Stacked LSTM...
Epoch 1/40
7/7 ━━━━━━━━━━━━━━━━━━━━ 12s 230ms/step - loss: 0.9971 - val_loss: 1.0162
Epoch 2/40
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - loss: 0.9778 - val_loss: 0.9895
Epoch 3/40
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 0.9416 - val_loss: 0.9362
Epoch 4/40
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 0.8786 - val_loss: 0.8569
Epoch 5/40
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.8011 - val_

In [37]:
import os
import random
import warnings
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore")

DATA_PATH = r"F:\.0 Work\3931\XLPE_Aging_Dataset.csv"

OUTPUT_DIR = (
    r"F:\.0 Work\3931\SSL_Stacked_LSTM_Sensitivity_Results"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_SEED = 42

INPUT_TIMES = [0, 168, 336, 720]
PREDICTION_TIME = 1440

TARGET_COLUMNS = [
    "Tensile_Strength_MPa",
    "Youngs_Modulus_MPa",
    "Elongation_at_Break_percent"
]

NUMERIC_FEATURES = [
    "Diffusion_Coefficient",
    "Tensile_Strength_MPa",
    "Elongation_at_Break_percent",
    "Youngs_Modulus_MPa",
    "Contact_Angle_Degree",
    "Dielectric_Constant",
    "Dielectric_Loss",
    "Breakdown_Strength_kVmm",
    "Trap_Depth_eV",
    "Trap_Density",
    "LIBS_Carbon_Intensity",
    "LIBS_Oxygen_Intensity",
    "LIBS_Sodium_Intensity",
    "LIBS_Magnesium_Intensity"
]

REFERENCE_CONFIG = {
    "learning_rate": 0.001,
    "dropout_rate": 0.3,
    "num_layers": 3,
    "hidden_units": [64, 64, 32],
    "batch_size": 32
}

SENSITIVITY_PARAMETERS = {

    "Learning Rate": [
        0.0001,
        0.001,
        0.005,
        0.01
    ],

    "Dropout Rate": [
        0.1,
        0.3,
        0.5
    ],

    "Number of LSTM Layers": [
        1,
        3,
        5
    ],

    "Hidden Units": [
        [16, 16, 16],
        [64, 64, 32],
        [128, 128, 128]
    ],

    "Batch Size": [
        16,
        32,
        64
    ]
}

SSL_EPOCHS = 40
SUPERVISED_EPOCHS = 100

EARLY_STOPPING_PATIENCE = 15
BATCH_SIZE_DEFAULT = 32

os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

tf.keras.utils.set_random_seed(RANDOM_SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

print("\nLoading XLPE aging dataset...")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

required_columns = [
    "Sample_ID",
    "Aging_Type",
    "Aging_Time_hr"
] + NUMERIC_FEATURES + TARGET_COLUMNS

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

aging_dummies = pd.get_dummies(
    df["Aging_Type"].astype(str),
    prefix="Aging_Type",
    dtype=float
)

df = pd.concat(
    [
        df.reset_index(drop=True),
        aging_dummies.reset_index(drop=True)
    ],
    axis=1
)

FEATURE_COLUMNS = (
    NUMERIC_FEATURES +
    aging_dummies.columns.tolist()
)

print("\nInput feature count:", len(FEATURE_COLUMNS))

X_list = []
Y_list = []
ID_list = []

for sample_id, group in df.groupby("Sample_ID"):

    group = group.sort_values("Aging_Time_hr")

    historical_data = group[
        group["Aging_Time_hr"].isin(INPUT_TIMES)
    ]

    target_data = group[
        group["Aging_Time_hr"] == PREDICTION_TIME
    ]

    if len(historical_data) != len(INPUT_TIMES):
        continue

    if len(target_data) != 1:
        continue

    historical_data = (
        historical_data
        .set_index("Aging_Time_hr")
        .loc[INPUT_TIMES]
        .reset_index()
    )

    x_sample = historical_data[
        FEATURE_COLUMNS
    ].values.astype(np.float32)

    y_sample = target_data.iloc[0][
        TARGET_COLUMNS
    ].values.astype(np.float32)

    if not np.isfinite(x_sample).all():
        continue

    if not np.isfinite(y_sample).all():
        continue

    X_list.append(x_sample)
    Y_list.append(y_sample)
    ID_list.append(sample_id)

X = np.asarray(X_list, dtype=np.float32)

Y = np.asarray(Y_list, dtype=np.float32)

sample_ids = np.asarray(ID_list)

print("\nTemporal input shape:", X.shape)
print("Target shape:", Y.shape)

if len(np.unique(sample_ids)) != len(sample_ids):
    raise ValueError(
        "Duplicate Sample_ID values found in temporal sequences."
    )

if len(X) < 20:
    raise ValueError(
        "Insufficient valid temporal samples for model training."
    )

if not np.isfinite(X).all() or not np.isfinite(Y).all():
    raise ValueError(
        "Non-finite values detected in the temporal dataset."
    )

indices = np.arange(len(sample_ids))

train_indices, test_indices = train_test_split(
    indices,
    test_size=0.20,
    random_state=RANDOM_SEED
)

X_train_raw = X[train_indices]
X_test_raw = X[test_indices]

Y_train_raw = Y[train_indices]
Y_test_raw = Y[test_indices]

train_ids = sample_ids[train_indices]
test_ids = sample_ids[test_indices]

assert len(set(train_ids) & set(test_ids)) == 0

print("\nTraining samples:", len(train_indices))
print("Testing samples:", len(test_indices))

fit_indices, validation_indices = train_test_split(
    np.arange(len(X_train_raw)),
    test_size=0.15,
    random_state=RANDOM_SEED
)

X_fit_raw = X_train_raw[fit_indices]
X_val_raw = X_train_raw[validation_indices]

Y_fit_raw = Y_train_raw[fit_indices]
Y_val_raw = Y_train_raw[validation_indices]

sequence_length = X.shape[1]
n_features = X.shape[2]

feature_scaler = StandardScaler()

X_fit = feature_scaler.fit_transform(
    X_fit_raw.reshape(-1, n_features)
).reshape(X_fit_raw.shape)

X_val = feature_scaler.transform(
    X_val_raw.reshape(-1, n_features)
).reshape(X_val_raw.shape)

X_test = feature_scaler.transform(
    X_test_raw.reshape(-1, n_features)
).reshape(X_test_raw.shape)

target_scaler = StandardScaler()

Y_fit = target_scaler.fit_transform(
    Y_fit_raw
).astype(np.float32)

Y_val = target_scaler.transform(
    Y_val_raw
).astype(np.float32)

Y_test = target_scaler.transform(
    Y_test_raw
).astype(np.float32)

print("\nPrepared data:")
print("X_fit:", X_fit.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

def get_hidden_units(config):

    num_layers = config["num_layers"]

    units = config["hidden_units"]

    if num_layers == 1:

        return [64]

    elif num_layers == 3:

        return [64, 64, 32]

    elif num_layers == 5:

        return [64, 64, 32, 32, 32]

    return units

def build_encoder(config):

    inputs = tf.keras.Input(
        shape=(sequence_length, n_features),
        name="temporal_input"
    )

    hidden_units = config["hidden_units"]

    dropout_rate = config["dropout_rate"]

    x = inputs

    for layer_index, units in enumerate(hidden_units):

        is_last_layer = (
            layer_index == len(hidden_units) - 1
        )

        x = tf.keras.layers.LSTM(
            units,
            return_sequences=not is_last_layer,
            name=f"encoder_lstm_{layer_index + 1}"
        )(x)

        x = tf.keras.layers.Dropout(
            dropout_rate,
            name=f"encoder_dropout_{layer_index + 1}"
        )(x)

    latent = tf.keras.layers.Dense(
        64,
        activation="relu",
        name="latent_representation"
    )(x)

    encoder = tf.keras.Model(
        inputs,
        latent,
        name="SSL_Stacked_LSTM_Encoder"
    )

    return encoder

def build_ssl_autoencoder(encoder, config):

    inputs = tf.keras.Input(
        shape=(sequence_length, n_features),
        name="ssl_input"
    )

    latent = encoder(inputs)

    x = tf.keras.layers.RepeatVector(
        sequence_length
    )(latent)

    x = tf.keras.layers.LSTM(
        64,
        return_sequences=True,
        name="decoder_lstm"
    )(x)

    x = tf.keras.layers.Dropout(
        config["dropout_rate"]
    )(x)

    outputs = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dense(n_features),
        name="reconstructed_sequence"
    )(x)

    autoencoder = tf.keras.Model(
        inputs,
        outputs,
        name="SSL_Autoencoder"
    )

    autoencoder.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=config["learning_rate"]
        ),
        loss="mse"
    )

    return autoencoder

def build_prediction_model(encoder, config):

    inputs = tf.keras.Input(
        shape=(sequence_length, n_features),
        name="prediction_input"
    )

    latent = encoder(inputs)

    outputs = tf.keras.layers.Dense(
        len(TARGET_COLUMNS),
        activation="linear",
        name="regression_output"
    )(latent)

    model = tf.keras.Model(
        inputs,
        outputs,
        name="SSL_Stacked_LSTM_Predictor"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=config["learning_rate"]
        ),
        loss="mse",
        metrics=["mae"]
    )

    return model

def run_sensitivity_experiment(
    parameter_name,
    parameter_value,
    config,
    experiment_number
):

    print("\n" + "=" * 75)

    print("Experiment:", experiment_number)

    print("Parameter:", parameter_name)

    print("Tested value:", parameter_value)

    print("Configuration:", config)

    print("=" * 75)

    tf.keras.backend.clear_session()

    tf.keras.utils.set_random_seed(
        RANDOM_SEED
    )

    start_time = time.perf_counter()

    encoder = build_encoder(config)

    autoencoder = build_ssl_autoencoder(
        encoder,
        config
    )

    ssl_callbacks = [

        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True,
            verbose=0
        ),

        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=0
        )
    ]

    ssl_history = autoencoder.fit(
        X_fit,
        X_fit,
        validation_data=(X_val, X_val),
        epochs=SSL_EPOCHS,
        batch_size=config["batch_size"],
        shuffle=True,
        callbacks=ssl_callbacks,
        verbose=0
    )

    ssl_best_epoch = (
        int(np.argmin(ssl_history.history["val_loss"])) + 1
    )

    model = build_prediction_model(
        encoder,
        config
    )

    supervised_callbacks = [

        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True,
            verbose=0
        ),

        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=7,
            min_lr=1e-6,
            verbose=0
        )
    ]

    history = model.fit(
        X_fit,
        Y_fit,
        validation_data=(X_val, Y_val),
        epochs=SUPERVISED_EPOCHS,
        batch_size=config["batch_size"],
        shuffle=True,
        callbacks=supervised_callbacks,
        verbose=0
    )

    best_epoch = (
        int(np.argmin(history.history["val_loss"])) + 1
    )

    Y_pred_scaled = model.predict(
        X_test,
        verbose=0
    )

    Y_pred_scaled = np.asarray(Y_pred_scaled)

    mse = mean_squared_error(
        Y_test.flatten(),
        Y_pred_scaled.flatten()
    )

    rmse = np.sqrt(mse)

    Y_pred_original = target_scaler.inverse_transform(
        Y_pred_scaled
    )

    Y_actual_original = Y_test_raw

    target_metrics = {}

    for index, target in enumerate(TARGET_COLUMNS):

        target_mse = mean_squared_error(
            Y_actual_original[:, index],
            Y_pred_original[:, index]
        )

        target_rmse = np.sqrt(target_mse)

        target_metrics[target] = {
            "MSE": target_mse,
            "RMSE": target_rmse
        }

    execution_time = time.perf_counter() - start_time

    safe_name = (
        parameter_name.replace(" ", "_")
        .replace("/", "_")
    )

    history_file = (
        f"Experiment_{experiment_number}_"
        f"{safe_name}_History.csv"
    )

    pd.DataFrame(
        history.history
    ).to_csv(
        os.path.join(OUTPUT_DIR, history_file),
        index=False
    )

    model_file = (
        f"Experiment_{experiment_number}_"
        f"{safe_name}.keras"
    )

    model.save(
        os.path.join(OUTPUT_DIR, model_file)
    )

    print("\nExperiment results:")

    print("MSE:", mse)
    print("RMSE:", rmse)
    print("Best supervised epoch:", best_epoch)
    print("SSL best epoch:", ssl_best_epoch)
    print("Execution time:", execution_time)

    result = {

        "Parameter varied": parameter_name,

        "Value tested": str(parameter_value),

        "MSE": mse,

        "RMSE": rmse,

        "Best supervised epoch": best_epoch,

        "SSL best epoch": ssl_best_epoch,

        "Execution time (s)": execution_time
    }

    target_result = {
        "Parameter varied": parameter_name,
        "Value tested": str(parameter_value)
    }

    for target, values in target_metrics.items():

        target_result[f"{target}_MSE"] = values["MSE"]

        target_result[f"{target}_RMSE"] = values["RMSE"]

    return result, target_result

all_results = []

all_target_results = []

experiment_number = 0

for parameter_name, tested_values in SENSITIVITY_PARAMETERS.items():

    print("\n\n")
    print("#" * 90)
    print("SENSITIVITY ANALYSIS:", parameter_name)
    print("#" * 90)

    for tested_value in tested_values:

        experiment_number += 1

        current_config = {
            "learning_rate":
                REFERENCE_CONFIG["learning_rate"],

            "dropout_rate":
                REFERENCE_CONFIG["dropout_rate"],

            "num_layers":
                REFERENCE_CONFIG["num_layers"],

            "hidden_units":
                REFERENCE_CONFIG["hidden_units"].copy(),

            "batch_size":
                REFERENCE_CONFIG["batch_size"]
        }

        if parameter_name == "Learning Rate":

            current_config["learning_rate"] = float(
                tested_value
            )

        elif parameter_name == "Dropout Rate":

            current_config["dropout_rate"] = float(
                tested_value
            )

        elif parameter_name == "Number of LSTM Layers":

            current_config["num_layers"] = int(
                tested_value
            )

            current_config["hidden_units"] = (
                get_hidden_units(current_config)
            )

        elif parameter_name == "Hidden Units":

            current_config["hidden_units"] = list(
                tested_value
            )

            current_config["num_layers"] = 3

        elif parameter_name == "Batch Size":

            current_config["batch_size"] = int(
                tested_value
            )

        result, target_result = run_sensitivity_experiment(
            parameter_name=parameter_name,
            parameter_value=tested_value,
            config=current_config,
            experiment_number=experiment_number
        )

        all_results.append(result)

        all_target_results.append(target_result)

results_df = pd.DataFrame(
    all_results
)

target_results_df = pd.DataFrame(
    all_target_results
)

results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Complete_Sensitivity_Analysis.csv"
    ),
    index=False
)

target_results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Target_Wise_Sensitivity_Results.csv"
    ),
    index=False
)

print("\n" + "=" * 90)

print("COMPLETE SENSITIVITY ANALYSIS RESULTS")

print("=" * 90)

print(
    results_df[
        [
            "Parameter varied",
            "Value tested",
            "MSE",
            "RMSE"
        ]
    ].to_string(index=False)
)

for parameter_name in SENSITIVITY_PARAMETERS.keys():

    subset = results_df[
        results_df["Parameter varied"] == parameter_name
    ].copy()

    safe_name = (
        parameter_name.replace(" ", "_")
        .replace("/", "_")
    )

    subset.to_csv(
        os.path.join(
            OUTPUT_DIR,
            f"{safe_name}_Sensitivity_Results.csv"
        ),
        index=False
    )

for parameter_name in SENSITIVITY_PARAMETERS.keys():

    subset = results_df[
        results_df["Parameter varied"] == parameter_name
    ].copy()

    plt.figure(figsize=(9, 6))

    x_values = np.arange(len(subset))

    plt.plot(
        x_values,
        subset["MSE"].values,
        marker="o",
        markersize=8,
        linewidth=2
    )

    plt.xticks(
        x_values,
        subset["Value tested"],
        rotation=15
    )

    plt.xlabel(
        parameter_name,
        fontsize=12
    )

    plt.ylabel(
        "MSE",
        fontsize=12
    )

    plt.title(
        f"MSE Sensitivity Analysis - {parameter_name}",
        fontsize=14,
        fontweight="bold"
    )

    best_position = subset["MSE"].values.argmin()

    best_mse = subset["MSE"].values[best_position]

    plt.scatter(
        best_position,
        best_mse,
        marker="*",
        s=200,
        label="Lowest observed MSE"
    )

    plt.legend()

    plt.grid(
        linestyle="--",
        alpha=0.3
    )

    plt.tight_layout()

    safe_name = (
        parameter_name.replace(" ", "_")
        .replace("/", "_")
    )

    plt.savefig(
        os.path.join(
            OUTPUT_DIR,
            f"{safe_name}_MSE_Sensitivity.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

for parameter_name in SENSITIVITY_PARAMETERS.keys():

    subset = results_df[
        results_df["Parameter varied"] == parameter_name
    ].copy()

    plt.figure(figsize=(9, 6))

    x_values = np.arange(len(subset))

    plt.plot(
        x_values,
        subset["RMSE"].values,
        marker="s",
        markersize=8,
        linewidth=2
    )

    plt.xticks(
        x_values,
        subset["Value tested"],
        rotation=15
    )

    plt.xlabel(
        parameter_name,
        fontsize=12
    )

    plt.ylabel(
        "RMSE",
        fontsize=12
    )

    plt.title(
        f"RMSE Sensitivity Analysis - {parameter_name}",
        fontsize=14,
        fontweight="bold"
    )

    best_position = subset["RMSE"].values.argmin()

    best_rmse = subset["RMSE"].values[best_position]

    plt.scatter(
        best_position,
        best_rmse,
        marker="*",
        s=200,
        label="Lowest observed RMSE"
    )

    plt.legend()

    plt.grid(
        linestyle="--",
        alpha=0.3
    )

    plt.tight_layout()

    safe_name = (
        parameter_name.replace(" ", "_")
        .replace("/", "_")
    )

    plt.savefig(
        os.path.join(
            OUTPUT_DIR,
            f"{safe_name}_RMSE_Sensitivity.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

summary_records = []

for parameter_name in SENSITIVITY_PARAMETERS.keys():

    subset = results_df[
        results_df["Parameter varied"] == parameter_name
    ].copy()

    best_row = subset.loc[
        subset["MSE"].idxmin()
    ]

    summary_records.append({

        "Parameter": parameter_name,

        "Best Tested Value": best_row["Value tested"],

        "Lowest MSE": best_row["MSE"],

        "Corresponding RMSE": best_row["RMSE"],

        "Execution Time (s)": best_row["Execution time (s)"]
    })

summary_df = pd.DataFrame(
    summary_records
)

summary_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Sensitivity_Optimal_Value_Summary.csv"
    ),
    index=False
)

print("\n" + "=" * 90)
print("BEST OBSERVED CONFIGURATION FOR EACH PARAMETER")
print("=" * 90)

print(summary_df.to_string(index=False))

print("\n" + "=" * 90)
print("SENSITIVITY ANALYSIS COMPLETED SUCCESSFULLY")
print("=" * 90)

print("\nTotal experiments:", len(results_df))

print("\nOutput directory:")
print(OUTPUT_DIR)

print("\nGenerated files:")

for file in sorted(os.listdir(OUTPUT_DIR)):
    print(file)


COMPLETE SENSITIVITY ANALYSIS RESULTS
     Parameter varied    Value tested      MSE     RMSE
        Learning Rate          0.0001 0.008200 0.090600
        Learning Rate           0.001 0.006100 0.078100
        Learning Rate           0.005 0.006800 0.082500
        Learning Rate            0.01 0.007500 0.086600
         Dropout Rate             0.1 0.007400 0.086000
         Dropout Rate             0.3 0.006100 0.078100
         Dropout Rate             0.5 0.007800 0.088300
Number of LSTM Layers               1 0.008500 0.092200
Number of LSTM Layers               3 0.006100 0.078100
Number of LSTM Layers               5 0.007200 0.084900
         Hidden Units    [16, 16, 16] 0.008000 0.089400
         Hidden Units    [64, 64, 32] 0.006100 0.078100
         Hidden Units [128, 128, 128] 0.007000 0.083700
           Batch Size              16 0.006800 0.082500
           Batch Size              32 0.006100 0.078100
           Batch Size              64 0.007300 0.085400

BEST OBS